# EdgeMLP + CatBoost Model

## We will be starting with the LI-Medium dataset

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import gc
from collections import Counter
import os
import kagglehub
import torch
import torch.nn as nn
from tqdm import trange
from sklearn.metrics import roc_auc_score, average_precision_score
!pip -q install catboost lightgbm
from catboost import CatBoostClassifier

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 27.3 MB/s eta 0:00:00


In [ ]:
kagglehub.dataset_download("ealtman2019/ibm-transactions-for-anti-money-laundering-aml", path="LI-Medium_Trans.csv")

100%|██████████| 2.77G/2.77G [02:51<00:00, 17.3MB/s]


'/root/.cache/kagglehub/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/versions/8/LI-Medium_Trans.csv'

In [ ]:
BASE = "/root/.cache/kagglehub/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/versions/8"
LI_Mtrans = pd.read_csv(f"{BASE}/LI-Medium_Trans.csv")
LI_Mtrans.head(5)

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:15,20,800104D70,20,800104D70,8095.07,US Dollar,8095.07,US Dollar,Reinvestment,0
1,2022/09/01 00:18,3196,800107150,3196,800107150,7739.29,US Dollar,7739.29,US Dollar,Reinvestment,0
2,2022/09/01 00:23,1208,80010E430,1208,80010E430,2654.22,US Dollar,2654.22,US Dollar,Reinvestment,0
3,2022/09/01 00:19,3203,80010EA80,3203,80010EA80,13284.41,US Dollar,13284.41,US Dollar,Reinvestment,0
4,2022/09/01 00:27,20,800104D20,20,800104D20,9.72,US Dollar,9.72,US Dollar,Reinvestment,0


In [3]:
def null_summary(df):
    null_counts = df.isna().sum()

    summary = pd.DataFrame({
        "null_count": null_counts,
    })

    return summary[summary["null_count"] > 0]

def duplicate_check(df):
    total = len(df)
    dup_all = df.duplicated().sum()

    return {
        "total_rows": total,
        "duplicate_rows": dup_all,
        "duplicate_rate": dup_all / total
    }

def timestamp_summary(df, time_col="Timestamp"):
    ts = pd.to_datetime(df[time_col], errors="coerce")

    return {
        "min_time": ts.min(),
        "max_time": ts.max(),
        "null_timestamps": ts.isna().sum()
    }

### Data Pre-processing

In [ ]:
LI_Mtrans["Is Laundering"].unique() #checking the cleanliness of the "Is Laundering" column; all good

array([0, 1])

In [ ]:
null_summary(LI_Mtrans) #checking for any null values; all good

,null_count


In [ ]:
duplicate_check(LI_Mtrans) #checking for any duplicates; 14 duplicates

{'total_rows': 31251483,
 'duplicate_rows': np.int64(14),
 'duplicate_rate': np.float64(4.4797874072088036e-07)}

In [ ]:
#dropping the duplicates
print(f"Shape before removing duplicates: {LI_Mtrans.shape}")
LI_Mtrans = LI_Mtrans.drop_duplicates().reset_index(drop=True)
print(f"New shape after removing duplicates: {LI_Mtrans.shape}")

Shape before removing duplicates: (31251483, 11)
New shape after removing duplicates: (31251469, 11)


In [ ]:
# Check for invalid account names
account_cols = [c for c in LI_Mtrans.columns if "account" in c.lower()]

if len(account_cols) == 0:
    print("No account columns found (no columns containing 'account').")
else:
    # Basic validity rules:
    # - not null
    # - not empty/whitespace
    # - only allows letters, digits, underscore, hyphen, dot (customize if needed)
    allowed_pattern = r"^[A-Za-z0-9_.-]+$"

    invalid_summary = {}

    for c in account_cols:
        s = LI_Mtrans[c].astype("string")

        is_null = s.isna()
        is_empty = s.str.strip().eq("")
        bad_chars = ~s.str.match(allowed_pattern, na=False)

        invalid_mask = is_null | is_empty | bad_chars
        invalid_count = int(invalid_mask.sum())

        invalid_summary[c] = {
            "invalid_count": invalid_count,
            "null_count": int(is_null.sum()),
            "empty_count": int(is_empty.sum()),
            "bad_char_count": int(bad_chars.sum())
        }

        print(f"\n[{c}] invalid rows: {invalid_count}")
        print("  null:", invalid_summary[c]["null_count"])
        print("  empty:", invalid_summary[c]["empty_count"])
        print("  bad_chars:", invalid_summary[c]["bad_char_count"])

        if invalid_count > 0:
            examples = LI_Mtrans.loc[invalid_mask, c].astype("string").head(10).tolist()
            print("  examples:", examples)

    print("\nChecked account columns:", account_cols)


[Account] invalid rows: 0
  null: 0
  empty: 0
  bad_chars: 0

[Account.1] invalid rows: 0
  null: 0
  empty: 0
  bad_chars: 0

Checked account columns: ['Account', 'Account.1']


In [ ]:
# Check for invalid transaction values

value_cols = [c for c in LI_Mtrans.columns if any(k in c.lower() for k in ["amount", "value"])]

if len(value_cols) == 0:
    print("No transaction value columns found (no columns containing 'amount' or 'value').")
else:
    for c in value_cols:
        x = pd.to_numeric(LI_Mtrans[c], errors="coerce")

        is_nan = x.isna()
        is_inf = np.isinf(x.to_numpy(dtype=float, copy=False))
        is_neg = x < 0
        is_zero = x == 0

        invalid_mask = is_nan | is_inf | is_neg

        print(f"\n[{c}]")
        print("  total rows:", len(LI_Mtrans))
        print("  NaN after numeric coercion:", int(is_nan.sum()))
        print("  inf:", int(is_inf.sum()))
        print("  negative:", int(is_neg.sum()))
        print("  zero:", int(is_zero.sum()))
        print("  invalid (NaN/inf/negative):", int(invalid_mask.sum()))

        if int(invalid_mask.sum()) > 0:
            example_rows = LI_Mtrans.loc[invalid_mask, [c]].head(10)
            print("  first invalid examples:")
            display(example_rows)

    print("\nChecked value columns:", value_cols)


[Amount Received]
  total rows: 31251469
  NaN after numeric coercion: 0
  inf: 0
  negative: 0
  zero: 0
  invalid (NaN/inf/negative): 0

[Amount Paid]
  total rows: 31251469
  NaN after numeric coercion: 0
  inf: 0
  negative: 0
  zero: 0
  invalid (NaN/inf/negative): 0

Checked value columns: ['Amount Received', 'Amount Paid']


In [ ]:
amt = pd.to_numeric(LI_Mtrans["Amount Received"], errors="coerce")
LI_Mtrans["Log Amount Received"] = np.log1p(amt)

In [ ]:
timestamp_summary(LI_Mtrans)

{'min_time': Timestamp('2022-09-01 00:00:00'),
 'max_time': Timestamp('2022-09-27 14:58:00'),
 'null_timestamps': np.int64(0)}

In [ ]:
# Setup for date data preprocessing

TS_COL = "Timestamp"
LABEL_COL = "Is Laundering"

TS_FORMAT = "%Y/%m/%d %H:%M"

# Use True for Large
USE_STREAMING = False
CHUNKSIZE = 2_000_000

CSV_PATH = "/root/.cache/kagglehub/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/versions/8/LI-Medium_Trans.csv"

def _coerce_label_to_int(series):
    return pd.to_numeric(series.astype(str).str.strip(), errors="coerce").fillna(0).astype(int)


def _time_aggs_from_df(df_in, ts_col=TS_COL, label_col=LABEL_COL, ts_format=TS_FORMAT):
    y = _coerce_label_to_int(df_in[label_col]).to_numpy(dtype=np.int8)

    if ts_format is None:
        ts = pd.to_datetime(df_in[ts_col], errors="coerce")
    else:
        ts = pd.to_datetime(df_in[ts_col], format=ts_format, errors="coerce")

    mask = ts.notna().to_numpy()
    bad_ts = int((~mask).sum())

    ts = ts[mask]
    y = y[mask]

    hour = ts.dt.hour.to_numpy(dtype=np.int16)
    dow = ts.dt.dayofweek.to_numpy(dtype=np.int16)  # Mon=0..Sun=6

    hour_total = np.bincount(hour, minlength=24).astype(np.int64)
    hour_pos   = np.bincount(hour, weights=y, minlength=24).astype(np.int64)

    dow_total = np.bincount(dow, minlength=7).astype(np.int64)
    dow_pos   = np.bincount(dow, weights=y, minlength=7).astype(np.int64)

    idx = dow * 24 + hour
    hd_total = np.bincount(idx, minlength=7*24).reshape(7, 24).astype(np.int64)
    hd_pos   = np.bincount(idx, weights=y, minlength=7*24).reshape(7, 24).astype(np.int64)

    return {
        "hour_total": hour_total, "hour_pos": hour_pos,
        "dow_total": dow_total,   "dow_pos": dow_pos,
        "hd_total": hd_total,     "hd_pos": hd_pos,
        "total_rows": int(mask.sum()),
        "bad_ts": bad_ts
    }


def _time_aggs_streaming(csv_path, chunksize=CHUNKSIZE, ts_col=TS_COL, label_col=LABEL_COL, ts_format=TS_FORMAT):
    hour_total = np.zeros(24, dtype=np.int64)
    hour_pos   = np.zeros(24, dtype=np.int64)

    dow_total  = np.zeros(7, dtype=np.int64)
    dow_pos    = np.zeros(7, dtype=np.int64)

    hd_total   = np.zeros((7, 24), dtype=np.int64)
    hd_pos     = np.zeros((7, 24), dtype=np.int64)

    total_rows = 0
    bad_ts = 0

    usecols = [ts_col, label_col]

    for chunk in pd.read_csv(csv_path, usecols=usecols, chunksize=chunksize):
        total_rows += len(chunk)

        y = _coerce_label_to_int(chunk[label_col]).to_numpy(dtype=np.int8)

        if ts_format is None:
            ts = pd.to_datetime(chunk[ts_col], errors="coerce")
        else:
            ts = pd.to_datetime(chunk[ts_col], format=ts_format, errors="coerce")

        mask = ts.notna().to_numpy()
        if not mask.all():
            bad_ts += int((~mask).sum())

        ts = ts[mask]
        y  = y[mask]

        hour = ts.dt.hour.to_numpy(dtype=np.int16)
        dow  = ts.dt.dayofweek.to_numpy(dtype=np.int16)

        hour_total += np.bincount(hour, minlength=24)
        hour_pos   += np.bincount(hour, weights=y, minlength=24).astype(np.int64)

        dow_total += np.bincount(dow, minlength=7)
        dow_pos   += np.bincount(dow, weights=y, minlength=7).astype(np.int64)

        idx = dow * 24 + hour
        flat_total = np.bincount(idx, minlength=7*24).reshape(7, 24)
        flat_pos   = np.bincount(idx, weights=y, minlength=7*24).reshape(7, 24).astype(np.int64)

        hd_total += flat_total
        hd_pos   += flat_pos

    return {
        "hour_total": hour_total, "hour_pos": hour_pos,
        "dow_total": dow_total,   "dow_pos": dow_pos,
        "hd_total": hd_total,     "hd_pos": hd_pos,
        "total_rows": int(total_rows),
        "bad_ts": int(bad_ts)
    }

In [ ]:
if USE_STREAMING:
    agg = _time_aggs_streaming(CSV_PATH, chunksize=CHUNKSIZE, ts_format=TS_FORMAT)
else:
    agg = _time_aggs_from_df(LI_Mtrans, ts_format=TS_FORMAT)

hour_total = agg["hour_total"]
hour_pos   = agg["hour_pos"]
dow_total  = agg["dow_total"]
dow_pos    = agg["dow_pos"]
hd_total   = agg["hd_total"]
hd_pos     = agg["hd_pos"]

print("Temporal EDA for:", "LI-Medium_Trans.csv")
print("Rows used (valid timestamps):", agg["total_rows"])
print("Bad/unparsed timestamps:", agg["bad_ts"])
print("Overall laundering rate:", float(hour_pos.sum() / max(hour_total.sum(), 1)))

Temporal EDA for: LI-Medium_Trans.csv
Rows used (valid timestamps): 31251469
Bad/unparsed timestamps: 0
Overall laundering rate: 0.000513287871363743


In [ ]:
hour_df = pd.DataFrame({
    "hour": np.arange(24),
    "tx_count": hour_total,
    "laundering_count": hour_pos,
    "laundering_rate": hour_pos / np.maximum(hour_total, 1)
})

dow_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
dow_df = pd.DataFrame({
    "dow": np.arange(7),
    "day": dow_names,
    "tx_count": dow_total,
    "laundering_count": dow_pos,
    "laundering_rate": dow_pos / np.maximum(dow_total, 1)
})

hd_rate = hd_pos / np.maximum(hd_total, 1)

In [ ]:
if TS_FORMAT is None:
    ts = pd.to_datetime(LI_Mtrans[TS_COL], errors="coerce")
else:
    ts = pd.to_datetime(LI_Mtrans[TS_COL], format=TS_FORMAT, errors="coerce")

LI_Mtrans["_ts"] = ts
LI_Mtrans["tx_hour"] = LI_Mtrans["_ts"].dt.hour.astype("Int16")
LI_Mtrans["tx_dow"] = LI_Mtrans["_ts"].dt.dayofweek.astype("Int16")     # Mon=0..Sun=6
LI_Mtrans["tx_month"] = LI_Mtrans["_ts"].dt.month.astype("Int16")
LI_Mtrans["tx_day"] = LI_Mtrans["_ts"].dt.day.astype("Int16")
LI_Mtrans["tx_date"] = LI_Mtrans["_ts"].dt.date                       # python date
LI_Mtrans["tx_is_weekend"] = LI_Mtrans["tx_dow"].isin([5, 6]).astype("Int8")
LI_Mtrans["tx_hour_sin"] = np.sin(2 * np.pi * LI_Mtrans["tx_hour"].fillna(0) / 24.0)
LI_Mtrans["tx_hour_cos"] = np.cos(2 * np.pi * LI_Mtrans["tx_hour"].fillna(0) / 24.0)

In [ ]:
LI_Mtrans.head()

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,...,Log Amount Received,_ts,tx_hour,tx_dow,tx_month,tx_day,tx_date,tx_is_weekend,tx_hour_sin,tx_hour_cos
0,2022/09/01 00:15,20,800104D70,20,800104D70,8095.07,US Dollar,8095.07,US Dollar,Reinvestment,...,8.999134,2022-09-01 00:15:00,0,3,9,1,2022-09-01,0,0.0,1.0
1,2022/09/01 00:18,3196,800107150,3196,800107150,7739.29,US Dollar,7739.29,US Dollar,Reinvestment,...,8.954194,2022-09-01 00:18:00,0,3,9,1,2022-09-01,0,0.0,1.0
2,2022/09/01 00:23,1208,80010E430,1208,80010E430,2654.22,US Dollar,2654.22,US Dollar,Reinvestment,...,7.884283,2022-09-01 00:23:00,0,3,9,1,2022-09-01,0,0.0,1.0
3,2022/09/01 00:19,3203,80010EA80,3203,80010EA80,13284.41,US Dollar,13284.41,US Dollar,Reinvestment,...,9.494422,2022-09-01 00:19:00,0,3,9,1,2022-09-01,0,0.0,1.0
4,2022/09/01 00:27,20,800104D20,20,800104D20,9.72,US Dollar,9.72,US Dollar,Reinvestment,...,2.372111,2022-09-01 00:27:00,0,3,9,1,2022-09-01,0,0.0,1.0


### Modeling

In [ ]:
df0 = LI_Mtrans

# Must exist from your preprocessing
assert "_ts" in df0.columns, "Expected LI_Mtrans['_ts'] from preprocessing."
assert "Is Laundering" in df0.columns, "Missing Is Laundering column."

# Keep only modeling columns (edit if you want more features)
keep_cols = [
    "_ts", "Is Laundering",
    "From Bank", "Account", "To Bank", "Account.1",
    "Amount Paid", "Amount Received",
    "Receiving Currency", "Payment Currency", "Payment Format",
    "Log Amount Received",
    "tx_hour", "tx_dow", "tx_is_weekend", "tx_hour_sin", "tx_hour_cos",
]
keep_cols = [c for c in keep_cols if c in df0.columns]
df = df0[keep_cols]

# Label clean (defensive)
y_ser = pd.to_numeric(df["Is Laundering"].astype(str).str.strip(), errors="coerce")
mask = y_ser.isin([0, 1])
df = df.loc[mask].copy()  # copy AFTER narrowing + filtering
df["Is Laundering"] = y_ser.loc[mask].astype(np.float32)

# Drop invalid timestamps
df = df.dropna(subset=["_ts"])

print("rows:", len(df), "pos_rate:", float(df["Is Laundering"].mean()))
print("columns:", df.columns.tolist())

rows: 31251469 pos_rate: 0.0005132879014126956
columns: ['_ts', 'Is Laundering', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Paid', 'Amount Received', 'Receiving Currency', 'Payment Currency', 'Payment Format', 'Log Amount Received', 'tx_hour', 'tx_dow', 'tx_is_weekend', 'tx_hour_sin', 'tx_hour_cos']


In [ ]:
order = np.argsort(df["_ts"].to_numpy())
df = df.iloc[order].reset_index(drop=True)
df["row_id"] = np.arange(len(df), dtype=np.int64)
print("time:", df["_ts"].min(), "->", df["_ts"].max())

time: 2022-09-01 00:00:00 -> 2022-09-27 14:58:00


In [ ]:
N = len(df)
n_train = int(0.70 * N)
n_val   = int(0.15 * N)

tr_idx_np   = np.arange(0, n_train, dtype=np.int64)
val_idx_np  = np.arange(n_train, n_train+n_val, dtype=np.int64)
test_idx_np = np.arange(n_train+n_val, N, dtype=np.int64)

print("Split sizes:", len(tr_idx_np), len(val_idx_np), len(test_idx_np))
print("Pos rates:",
      float(df["Is Laundering"].to_numpy()[tr_idx_np].mean()),
      float(df["Is Laundering"].to_numpy()[val_idx_np].mean()),
      float(df["Is Laundering"].to_numpy()[test_idx_np].mean()))

Split sizes: 21876028 4687720 4687721
Pos rates: 0.0004867885436397046 0.0005256286822259426 0.0006246105767786503


In [ ]:
MAX_EDGES = 3_000_000  # adjust upward later if stable

if len(df) > MAX_EDGES:
    df_edge = df.iloc[:MAX_EDGES].copy()
else:
    df_edge = df.copy()

print("EdgeMLP edges used:", len(df_edge), "of", len(df))

EdgeMLP edges used: 3000000 of 31251469


In [ ]:
src_df = df_edge[["From Bank","Account"]]
dst_df = df_edge[["To Bank","Account.1"]].rename(columns={"To Bank":"From Bank","Account.1":"Account"})
all_df = pd.concat([src_df, dst_df], ignore_index=True)

codes, uniques = pd.factorize(list(map(tuple, all_df.to_numpy())))
m = len(df_edge)

src = codes[:m].astype(np.int64)
dst = codes[m:].astype(np.int64)
y   = df_edge["Is Laundering"].to_numpy(dtype=np.float32)

num_nodes = len(uniques)
print("nodes:", num_nodes, "edges:", m, "pos_rate:", float(y.mean()))

/tmp/ipython-input-2499473087.py:5: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  codes, uniques = pd.factorize(list(map(tuple, all_df.to_numpy())))


nodes: 1523875 edges: 3000000 pos_rate: 0.0001656666718190536


In [ ]:
class EdgeMLP(nn.Module):
    def __init__(self, n_nodes, d=128, dropout=0.2):
        super().__init__()
        self.emb = nn.Embedding(n_nodes, d)
        self.drop = nn.Dropout(dropout)
        self.net = nn.Sequential(
            nn.Linear(4*d, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, s, t):
        Hs = self.drop(self.emb(s))
        Hd = self.drop(self.emb(t))
        x = torch.cat([Hs, Hd, (Hs - Hd).abs(), Hs * Hd], dim=1)
        return self.net(x).squeeze(-1)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

src_gpu = torch.from_numpy(src).long().to(device)
dst_gpu = torch.from_numpy(dst).long().to(device)
y_gpu   = torch.from_numpy(y).float().to(device)

BATCH  = 262_144
EPOCHS = 5  # start small on CPU; raise later if stable
N_FOLDS = 5

def fit_edgemlp(train_idx_np, epochs=EPOCHS):
    model = EdgeMLP(num_nodes, d=128, dropout=0.2).to(device)

    pos_rate = float(y[train_idx_np].mean())
    pos_w = (1.0 - pos_rate) / max(pos_rate, 1e-12)
    pos_weight = torch.tensor([min(pos_w, 1000.0)], device=device)  # cap helps stability
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

    for ep in range(epochs):
        model.train()
        order = train_idx_np.copy()
        np.random.shuffle(order)

        running = 0.0
        for i in range(0, len(order), BATCH):
            b_np = order[i:i+BATCH]
            b = torch.from_numpy(b_np).to(device)

            opt.zero_grad(set_to_none=True)
            logits = model(src_gpu[b], dst_gpu[b])
            loss = loss_fn(logits, y_gpu[b])
            loss.backward()
            opt.step()

            running += float(loss.detach()) * len(b_np)

        print(f"EdgeMLP Epoch {ep:02d} | loss={running/len(order):.6f}")

    return model

@torch.no_grad()
def predict_edgemlp(model, idx_np):
    model.eval()
    out = np.empty(len(idx_np), dtype=np.float32)
    for i in range(0, len(idx_np), BATCH):
        b_np = idx_np[i:i+BATCH]
        b = torch.from_numpy(b_np).to(device)
        out[i:i+len(b_np)] = torch.sigmoid(model(src_gpu[b], dst_gpu[b])).cpu().numpy().astype(np.float32)
    return out

device: cuda


In [ ]:
N_edge = len(df_edge)

# Time split indices for df_edge
n_train_edge = min(int(0.70 * len(df)), N_edge)  # align with original split boundary
n_val_edge   = min(int(0.15 * len(df)), max(0, N_edge - n_train_edge))

tr_edge = np.arange(0, n_train_edge, dtype=np.int64)
va_edge = np.arange(n_train_edge, n_train_edge + n_val_edge, dtype=np.int64)
te_edge = np.arange(n_train_edge + n_val_edge, N_edge, dtype=np.int64)

print("EdgeMLP split sizes:", len(tr_edge), len(va_edge), len(te_edge))
print("EdgeMLP pos rates:", float(y[tr_edge].mean()), float(y[va_edge].mean()) if len(va_edge)>0 else None)

edge_score_edge = np.full(N_edge, np.nan, dtype=np.float32)

# Contiguous time blocks within TRAIN for OOF
blocks = np.array_split(tr_edge, N_FOLDS)

print("OOF EdgeMLP scores on TRAIN (time-respecting blocks)...")
for k in range(N_FOLDS):
    holdout = blocks[k]

    # SAFE FIX: skip first block (no earlier data)
    if k == 0:
        print(f"Block {k+1}/{N_FOLDS}: skipped (no earlier data).")
        continue

    train_f = np.concatenate(blocks[:k])  # now guaranteed non-empty

    m_k = fit_edgemlp(train_f, epochs=EPOCHS)
    edge_score_edge[holdout] = predict_edgemlp(m_k, holdout)

    print(f"Block {k+1}/{N_FOLDS} done.")

print("Final EdgeMLP on full TRAIN → score VAL/TEST + fill skipped TRAIN block if any...")
m_final = fit_edgemlp(tr_edge, epochs=EPOCHS)

# Fill any skipped earliest block (optional; see note below)
nan_tr = tr_edge[np.isnan(edge_score_edge[tr_edge])]
if len(nan_tr) > 0:
    edge_score_edge[nan_tr] = predict_edgemlp(m_final, nan_tr)

if len(va_edge) > 0:
    edge_score_edge[va_edge] = predict_edgemlp(m_final, va_edge)
if len(te_edge) > 0:
    edge_score_edge[te_edge] = predict_edgemlp(m_final, te_edge)

print("NaNs remaining (TRAIN/VAL/TEST):",
      np.isnan(edge_score_edge[tr_edge]).sum(),
      np.isnan(edge_score_edge[va_edge]).sum() if len(va_edge)>0 else 0,
      np.isnan(edge_score_edge[te_edge]).sum() if len(te_edge)>0 else 0)

# Attach edge scores back to df_edge and then map into df (full) by row order
df_edge["edge_score"] = edge_score_edge

EdgeMLP split sizes: 3000000 0 0
EdgeMLP pos rates: 0.0001656666718190536 None
OOF EdgeMLP scores on TRAIN (time-respecting blocks)...
Block 1/5: skipped (no earlier data).
EdgeMLP Epoch 00 | loss=0.545733
EdgeMLP Epoch 01 | loss=0.190356
EdgeMLP Epoch 02 | loss=0.138347
EdgeMLP Epoch 03 | loss=0.128018
EdgeMLP Epoch 04 | loss=0.096306
Block 2/5 done.
EdgeMLP Epoch 00 | loss=0.437829
EdgeMLP Epoch 01 | loss=0.158902
EdgeMLP Epoch 02 | loss=0.135440
EdgeMLP Epoch 03 | loss=0.099800
EdgeMLP Epoch 04 | loss=0.093762
Block 3/5 done.
EdgeMLP Epoch 00 | loss=0.450298
EdgeMLP Epoch 01 | loss=0.274491
EdgeMLP Epoch 02 | loss=0.237199
EdgeMLP Epoch 03 | loss=0.226210
EdgeMLP Epoch 04 | loss=0.211470
Block 4/5 done.
EdgeMLP Epoch 00 | loss=0.486986
EdgeMLP Epoch 01 | loss=0.307122
EdgeMLP Epoch 02 | loss=0.286354
EdgeMLP Epoch 03 | loss=0.280946
EdgeMLP Epoch 04 | loss=0.271713
Block 5/5 done.
Final EdgeMLP on full TRAIN → score VAL/TEST + fill skipped TRAIN block if any...
EdgeMLP Epoch 00 | lo

In [ ]:
df["edge_score"] = np.nan

# Because df_edge is the first MAX_EDGES rows of df (time-sorted), row alignment is direct:
df.loc[:len(df_edge)-1, "edge_score"] = df_edge["edge_score"].to_numpy()

print("edge_score filled rows:", df["edge_score"].notna().sum(), "out of", len(df))

edge_score filled rows: 3000000 out of 31251469


In [ ]:
df["Amount Paid"] = pd.to_numeric(df["Amount Paid"], errors="coerce").fillna(0.0)
df["Amount Received"] = pd.to_numeric(df["Amount Received"], errors="coerce").fillna(0.0)

# Use your Log Amount Received if present
if "Log Amount Received" in df.columns:
    df["log_amt_recv"] = pd.to_numeric(df["Log Amount Received"], errors="coerce")
else:
    df["log_amt_recv"] = np.log1p(df["Amount Received"].astype(np.float32))

df["log_amt_paid"] = np.log1p(df["Amount Paid"].astype(np.float32))
df["cross_bank"] = (df["From Bank"].astype(str) != df["To Bank"].astype(str)).astype(np.int8)

# Minimal CatBoost feature set (edit as needed)
cat_cols = ["From Bank", "To Bank", "Receiving Currency", "Payment Currency", "Payment Format"]
num_cols = ["edge_score", "log_amt_paid", "log_amt_recv", "cross_bank",
            "tx_hour", "tx_dow", "tx_is_weekend", "tx_hour_sin", "tx_hour_cos"]

feature_cols = cat_cols + num_cols

# Drop rows where edge_score is missing (rows beyond MAX_EDGES or skipped fold)
df_cb = df.dropna(subset=["edge_score"]).copy()

X = df_cb[feature_cols].copy()
y_cb = df_cb["Is Laundering"].astype(int).to_numpy(np.int32)

# Recompute splits in df_cb by time order position (still time-respecting, but on the filtered df_cb)
n = len(df_cb)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)

Xtr, ytr = X.iloc[:n_train], y_cb[:n_train]
Xva, yva = X.iloc[n_train:n_train+n_val], y_cb[n_train:n_train+n_val]
Xte, yte = X.iloc[n_train+n_val:], y_cb[n_train+n_val:]

print("CatBoost sizes:", Xtr.shape, Xva.shape, Xte.shape)
print("CatBoost pos rates:", ytr.mean(), yva.mean(), yte.mean())

CatBoost sizes: (2100000, 14) (450000, 14) (450000, 14)
CatBoost pos rates: 0.00012238095238095237 0.00025111111111111113 0.00028222222222222223


In [ ]:
cat_idx = [Xtr.columns.get_loc(c) for c in cat_cols]

cb = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.05,
    depth=8,
    loss_function="Logloss",
    eval_metric="PRAUC",
    random_seed=0,
    verbose=200,
    auto_class_weights="Balanced",
    task_type="CPU"
)

cb.fit(Xtr, ytr, eval_set=(Xva, yva), cat_features=cat_idx, use_best_model=True)

pva = cb.predict_proba(Xva)[:, 1]
pte = cb.predict_proba(Xte)[:, 1]

print("\n=== EdgeMLP → CatBoost (time split, leak-free stacking) ===")
print("VAL  ROC:", roc_auc_score(yva, pva))
print("VAL  PR :", average_precision_score(yva, pva))
print("TEST ROC:", roc_auc_score(yte, pte))
print("TEST PR :", average_precision_score(yte, pte))

0:	learn: 0.8523418	test: 0.8791125	best: 0.8791125 (0)	total: 470ms	remaining: 23m 28s
200:	learn: 0.9893425	test: 0.9530408	best: 0.9536158 (165)	total: 2m 38s	remaining: 36m 44s
400:	learn: 0.9989118	test: 0.9510014	best: 0.9536158 (165)	total: 5m 55s	remaining: 38m 25s
600:	learn: 0.9999615	test: 0.9487821	best: 0.9536158 (165)	total: 9m 58s	remaining: 39m 50s
800:	learn: 0.9999966	test: 0.9459441	best: 0.9536158 (165)	total: 14m 13s	remaining: 39m 3s
1000:	learn: 0.9999987	test: 0.9442265	best: 0.9536158 (165)	total: 18m 17s	remaining: 36m 30s
1200:	learn: 0.9999988	test: 0.9439893	best: 0.9536158 (165)	total: 21m 31s	remaining: 32m 14s
1400:	learn: 0.9999988	test: 0.9439874	best: 0.9536158 (165)	total: 24m 3s	remaining: 27m 27s
1600:	learn: 0.9999988	test: 0.9439874	best: 0.9536158 (165)	total: 26m 7s	remaining: 22m 49s
1800:	learn: 0.9999989	test: 0.9429589	best: 0.9536158 (165)	total: 28m 50s	remaining: 19m 11s
2000:	learn: 0.9999989	test: 0.9426839	best: 0.9536158 (165)	total:

In [ ]:
DATASET_NAME = "LI-Medium"          # <-- change each run
MODEL_NAME   = "EdgeMLP+CatBoost"
SPLIT_NAME   = "time_70_15_15"

val_roc = roc_auc_score(yva, pva)
test_roc = roc_auc_score(yte, pte)

val_pr = average_precision_score(yva, pva)
test_pr = average_precision_score(yte, pte)

val_base = float(yva.mean())
test_base = float(yte.mean())

row = {
    "dataset": DATASET_NAME,
    "model": MODEL_NAME,
    "split": SPLIT_NAME,
    "val_roc": val_roc,
    "test_roc": test_roc,
    "val_pr_auc": val_pr,
    "test_pr_auc": test_pr,
    "val_base_rate": val_base,
    "test_base_rate": test_base,
    "val_pr_lift": val_pr / val_base if val_base > 0 else None,
    "test_pr_lift": test_pr / test_base if test_base > 0 else None,
    "n_val": int(len(yva)),
    "n_test": int(len(yte)),
}

new_df = pd.DataFrame([row])

path = Path("aml_model_results.csv")
if path.exists():
    old = pd.read_csv(path)
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(path, index=False)
print(f"Saved → {path} (rows={len(out)})")
out.tail(10)

Saved → aml_model_results.csv (rows=1)


,dataset,model,split,val_roc,test_roc,val_pr_auc,test_pr_auc,val_base_rate,test_base_rate,val_pr_lift,test_pr_lift,n_val,n_test
0,LI-Medium,EdgeMLP+CatBoost,time_70_15_15,0.945931,0.94683,0.003395,0.003856,0.000251,0.000282,13.520729,13.663972,450000,450000


## Next, we will be using the HI-Medium dataset

In [ ]:
kagglehub.dataset_download("ealtman2019/ibm-transactions-for-anti-money-laundering-aml", path="HI-Medium_Trans.csv")

Using Colab cache for faster access to the 'ibm-transactions-for-anti-money-laundering-aml' dataset.


'/kaggle/input/ibm-transactions-for-anti-money-laundering-aml/HI-Medium_Trans.csv'

In [ ]:
BASE = "/kaggle/input/ibm-transactions-for-anti-money-laundering-aml"
HI_Mtrans = pd.read_csv(f"{BASE}/HI-Medium_Trans.csv")
HI_Mtrans.head(5)

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:17,20,800104D70,20,800104D70,6794.63,US Dollar,6794.63,US Dollar,Reinvestment,0
1,2022/09/01 00:02,3196,800107150,3196,800107150,7739.29,US Dollar,7739.29,US Dollar,Reinvestment,0
2,2022/09/01 00:17,1208,80010E430,1208,80010E430,1880.23,US Dollar,1880.23,US Dollar,Reinvestment,0
3,2022/09/01 00:03,1208,80010E650,20,80010E6F0,73966883.00,US Dollar,73966883.00,US Dollar,Cheque,0
4,2022/09/01 00:02,1208,80010E650,20,80010EA30,45868454.00,US Dollar,45868454.00,US Dollar,Cheque,0


In [ ]:
HI_Mtrans["Is Laundering"].unique() #checking the cleanliness of the "Is Laundering" column; all good

array([0, 1])

In [ ]:
null_summary(HI_Mtrans) #checking for any null values; all good

,null_count


In [ ]:
duplicate_check(HI_Mtrans) #checking for any duplicates; 20 duplicates

{'total_rows': 31898238,
 'duplicate_rows': np.int64(20),
 'duplicate_rate': np.float64(6.269938797246419e-07)}

In [ ]:
#dropping the duplicates
print(f"Shape before removing duplicates: {HI_Mtrans.shape}")
HI_Mtrans = HI_Mtrans.drop_duplicates().reset_index(drop=True)
print(f"New shape after removing duplicates: {HI_Mtrans.shape}")

Shape before removing duplicates: (31898238, 11)
New shape after removing duplicates: (31898218, 11)


In [ ]:
# Check for invalid account names
account_cols = [c for c in HI_Mtrans.columns if "account" in c.lower()]

if len(account_cols) == 0:
    print("No account columns found (no columns containing 'account').")
else:
    # Basic validity rules:
    # - not null
    # - not empty/whitespace
    # - only allows letters, digits, underscore, hyphen, dot (customize if needed)
    allowed_pattern = r"^[A-Za-z0-9_.-]+$"

    invalid_summary = {}

    for c in account_cols:
        s = HI_Mtrans[c].astype("string")

        is_null = s.isna()
        is_empty = s.str.strip().eq("")
        bad_chars = ~s.str.match(allowed_pattern, na=False)

        invalid_mask = is_null | is_empty | bad_chars
        invalid_count = int(invalid_mask.sum())

        invalid_summary[c] = {
            "invalid_count": invalid_count,
            "null_count": int(is_null.sum()),
            "empty_count": int(is_empty.sum()),
            "bad_char_count": int(bad_chars.sum())
        }

        print(f"\n[{c}] invalid rows: {invalid_count}")
        print("  null:", invalid_summary[c]["null_count"])
        print("  empty:", invalid_summary[c]["empty_count"])
        print("  bad_chars:", invalid_summary[c]["bad_char_count"])

        if invalid_count > 0:
            examples = HI_Mtrans.loc[invalid_mask, c].astype("string").head(10).tolist()
            print("  examples:", examples)

    print("\nChecked account columns:", account_cols)


[Account] invalid rows: 0
  null: 0
  empty: 0
  bad_chars: 0

[Account.1] invalid rows: 0
  null: 0
  empty: 0
  bad_chars: 0

Checked account columns: ['Account', 'Account.1']


In [ ]:
# Check for invalid transaction values

value_cols = [c for c in HI_Mtrans.columns if any(k in c.lower() for k in ["amount", "value"])]

if len(value_cols) == 0:
    print("No transaction value columns found (no columns containing 'amount' or 'value').")
else:
    for c in value_cols:
        x = pd.to_numeric(HI_Mtrans[c], errors="coerce")

        is_nan = x.isna()
        is_inf = np.isinf(x.to_numpy(dtype=float, copy=False))
        is_neg = x < 0
        is_zero = x == 0

        invalid_mask = is_nan | is_inf | is_neg

        print(f"\n[{c}]")
        print("  total rows:", len(HI_Mtrans))
        print("  NaN after numeric coercion:", int(is_nan.sum()))
        print("  inf:", int(is_inf.sum()))
        print("  negative:", int(is_neg.sum()))
        print("  zero:", int(is_zero.sum()))
        print("  invalid (NaN/inf/negative):", int(invalid_mask.sum()))

        if int(invalid_mask.sum()) > 0:
            example_rows = HI_Mtrans.loc[invalid_mask, [c]].head(10)
            print("  first invalid examples:")
            display(example_rows)

    print("\nChecked value columns:", value_cols)


[Amount Received]
  total rows: 31898218
  NaN after numeric coercion: 0
  inf: 0
  negative: 0
  zero: 0
  invalid (NaN/inf/negative): 0

[Amount Paid]
  total rows: 31898218
  NaN after numeric coercion: 0
  inf: 0
  negative: 0
  zero: 0
  invalid (NaN/inf/negative): 0

Checked value columns: ['Amount Received', 'Amount Paid']


In [ ]:
amt = pd.to_numeric(HI_Mtrans["Amount Received"], errors="coerce")
HI_Mtrans["Log Amount Received"] = np.log1p(amt)

In [ ]:
timestamp_summary(HI_Mtrans)

{'min_time': Timestamp('2022-09-01 00:00:00'),
 'max_time': Timestamp('2022-09-28 15:58:00'),
 'null_timestamps': np.int64(0)}

In [ ]:
# Setup for date data preprocessing

TS_COL = "Timestamp"
LABEL_COL = "Is Laundering"

TS_FORMAT = "%Y/%m/%d %H:%M"

# Use True for Large
USE_STREAMING = False
CHUNKSIZE = 2_000_000

CSV_PATH = "/kaggle/input/ibm-transactions-for-anti-money-laundering-aml/HI-Medium_Trans.csv"

def _coerce_label_to_int(series):
    return pd.to_numeric(series.astype(str).str.strip(), errors="coerce").fillna(0).astype(int)


def _time_aggs_from_df(df_in, ts_col=TS_COL, label_col=LABEL_COL, ts_format=TS_FORMAT):
    y = _coerce_label_to_int(df_in[label_col]).to_numpy(dtype=np.int8)

    if ts_format is None:
        ts = pd.to_datetime(df_in[ts_col], errors="coerce")
    else:
        ts = pd.to_datetime(df_in[ts_col], format=ts_format, errors="coerce")

    mask = ts.notna().to_numpy()
    bad_ts = int((~mask).sum())

    ts = ts[mask]
    y = y[mask]

    hour = ts.dt.hour.to_numpy(dtype=np.int16)
    dow = ts.dt.dayofweek.to_numpy(dtype=np.int16)  # Mon=0..Sun=6

    hour_total = np.bincount(hour, minlength=24).astype(np.int64)
    hour_pos   = np.bincount(hour, weights=y, minlength=24).astype(np.int64)

    dow_total = np.bincount(dow, minlength=7).astype(np.int64)
    dow_pos   = np.bincount(dow, weights=y, minlength=7).astype(np.int64)

    idx = dow * 24 + hour
    hd_total = np.bincount(idx, minlength=7*24).reshape(7, 24).astype(np.int64)
    hd_pos   = np.bincount(idx, weights=y, minlength=7*24).reshape(7, 24).astype(np.int64)

    return {
        "hour_total": hour_total, "hour_pos": hour_pos,
        "dow_total": dow_total,   "dow_pos": dow_pos,
        "hd_total": hd_total,     "hd_pos": hd_pos,
        "total_rows": int(mask.sum()),
        "bad_ts": bad_ts
    }


def _time_aggs_streaming(csv_path, chunksize=CHUNKSIZE, ts_col=TS_COL, label_col=LABEL_COL, ts_format=TS_FORMAT):
    hour_total = np.zeros(24, dtype=np.int64)
    hour_pos   = np.zeros(24, dtype=np.int64)

    dow_total  = np.zeros(7, dtype=np.int64)
    dow_pos    = np.zeros(7, dtype=np.int64)

    hd_total   = np.zeros((7, 24), dtype=np.int64)
    hd_pos     = np.zeros((7, 24), dtype=np.int64)

    total_rows = 0
    bad_ts = 0

    usecols = [ts_col, label_col]

    for chunk in pd.read_csv(csv_path, usecols=usecols, chunksize=chunksize):
        total_rows += len(chunk)

        y = _coerce_label_to_int(chunk[label_col]).to_numpy(dtype=np.int8)

        if ts_format is None:
            ts = pd.to_datetime(chunk[ts_col], errors="coerce")
        else:
            ts = pd.to_datetime(chunk[ts_col], format=ts_format, errors="coerce")

        mask = ts.notna().to_numpy()
        if not mask.all():
            bad_ts += int((~mask).sum())

        ts = ts[mask]
        y  = y[mask]

        hour = ts.dt.hour.to_numpy(dtype=np.int16)
        dow  = ts.dt.dayofweek.to_numpy(dtype=np.int16)

        hour_total += np.bincount(hour, minlength=24)
        hour_pos   += np.bincount(hour, weights=y, minlength=24).astype(np.int64)

        dow_total += np.bincount(dow, minlength=7)
        dow_pos   += np.bincount(dow, weights=y, minlength=7).astype(np.int64)

        idx = dow * 24 + hour
        flat_total = np.bincount(idx, minlength=7*24).reshape(7, 24)
        flat_pos   = np.bincount(idx, weights=y, minlength=7*24).reshape(7, 24).astype(np.int64)

        hd_total += flat_total
        hd_pos   += flat_pos

    return {
        "hour_total": hour_total, "hour_pos": hour_pos,
        "dow_total": dow_total,   "dow_pos": dow_pos,
        "hd_total": hd_total,     "hd_pos": hd_pos,
        "total_rows": int(total_rows),
        "bad_ts": int(bad_ts)
    }

In [ ]:
if USE_STREAMING:
    agg = _time_aggs_streaming(CSV_PATH, chunksize=CHUNKSIZE, ts_format=TS_FORMAT)
else:
    agg = _time_aggs_from_df(HI_Mtrans, ts_format=TS_FORMAT)

hour_total = agg["hour_total"]
hour_pos   = agg["hour_pos"]
dow_total  = agg["dow_total"]
dow_pos    = agg["dow_pos"]
hd_total   = agg["hd_total"]
hd_pos     = agg["hd_pos"]

print("Temporal EDA for:", "HI-Medium_Trans.csv")
print("Rows used (valid timestamps):", agg["total_rows"])
print("Bad/unparsed timestamps:", agg["bad_ts"])
print("Overall laundering rate:", float(hour_pos.sum() / max(hour_total.sum(), 1)))

Temporal EDA for: HI-Medium_Trans.csv
Rows used (valid timestamps): 31898218
Bad/unparsed timestamps: 0
Overall laundering rate: 0.0011044504116186052


In [ ]:
hour_df = pd.DataFrame({
    "hour": np.arange(24),
    "tx_count": hour_total,
    "laundering_count": hour_pos,
    "laundering_rate": hour_pos / np.maximum(hour_total, 1)
})

dow_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
dow_df = pd.DataFrame({
    "dow": np.arange(7),
    "day": dow_names,
    "tx_count": dow_total,
    "laundering_count": dow_pos,
    "laundering_rate": dow_pos / np.maximum(dow_total, 1)
})

hd_rate = hd_pos / np.maximum(hd_total, 1)

In [ ]:
if TS_FORMAT is None:
    ts = pd.to_datetime(HI_Mtrans[TS_COL], errors="coerce")
else:
    ts = pd.to_datetime(HI_Mtrans[TS_COL], format=TS_FORMAT, errors="coerce")

HI_Mtrans["_ts"] = ts
HI_Mtrans["tx_hour"] = HI_Mtrans["_ts"].dt.hour.astype("Int16")
HI_Mtrans["tx_dow"] = HI_Mtrans["_ts"].dt.dayofweek.astype("Int16")     # Mon=0..Sun=6
HI_Mtrans["tx_month"] = HI_Mtrans["_ts"].dt.month.astype("Int16")
HI_Mtrans["tx_day"] = HI_Mtrans["_ts"].dt.day.astype("Int16")
HI_Mtrans["tx_date"] = HI_Mtrans["_ts"].dt.date                       # python date
HI_Mtrans["tx_is_weekend"] = HI_Mtrans["tx_dow"].isin([5, 6]).astype("Int8")
HI_Mtrans["tx_hour_sin"] = np.sin(2 * np.pi * HI_Mtrans["tx_hour"].fillna(0) / 24.0)
HI_Mtrans["tx_hour_cos"] = np.cos(2 * np.pi * HI_Mtrans["tx_hour"].fillna(0) / 24.0)

In [ ]:
HI_Mtrans.head()

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,...,Log Amount Received,_ts,tx_hour,tx_dow,tx_month,tx_day,tx_date,tx_is_weekend,tx_hour_sin,tx_hour_cos
0,2022/09/01 00:17,20,800104D70,20,800104D70,6794.63,US Dollar,6794.63,US Dollar,Reinvestment,...,8.824035,2022-09-01 00:17:00,0,3,9,1,2022-09-01,0,0.0,1.0
1,2022/09/01 00:02,3196,800107150,3196,800107150,7739.29,US Dollar,7739.29,US Dollar,Reinvestment,...,8.954194,2022-09-01 00:02:00,0,3,9,1,2022-09-01,0,0.0,1.0
2,2022/09/01 00:17,1208,80010E430,1208,80010E430,1880.23,US Dollar,1880.23,US Dollar,Reinvestment,...,7.539681,2022-09-01 00:17:00,0,3,9,1,2022-09-01,0,0.0,1.0
3,2022/09/01 00:03,1208,80010E650,20,80010E6F0,73966883.00,US Dollar,73966883.00,US Dollar,Cheque,...,18.119128,2022-09-01 00:03:00,0,3,9,1,2022-09-01,0,0.0,1.0
4,2022/09/01 00:02,1208,80010E650,20,80010EA30,45868454.00,US Dollar,45868454.00,US Dollar,Cheque,...,17.641288,2022-09-01 00:02:00,0,3,9,1,2022-09-01,0,0.0,1.0


### Modeling

In [ ]:
df0 = HI_Mtrans

# Must exist from your preprocessing
assert "_ts" in df0.columns, "Expected HI_Mtrans['_ts'] from preprocessing."
assert "Is Laundering" in df0.columns, "Missing Is Laundering column."

# Keep only modeling columns (edit if you want more features)
keep_cols = [
    "_ts", "Is Laundering",
    "From Bank", "Account", "To Bank", "Account.1",
    "Amount Paid", "Amount Received",
    "Receiving Currency", "Payment Currency", "Payment Format",
    "Log Amount Received",
    "tx_hour", "tx_dow", "tx_is_weekend", "tx_hour_sin", "tx_hour_cos",
]
keep_cols = [c for c in keep_cols if c in df0.columns]
df = df0[keep_cols]

# Label clean (defensive)
y_ser = pd.to_numeric(df["Is Laundering"].astype(str).str.strip(), errors="coerce")
mask = y_ser.isin([0, 1])
df = df.loc[mask].copy()  # copy AFTER narrowing + filtering
df["Is Laundering"] = y_ser.loc[mask].astype(np.float32)

# Drop invalid timestamps
df = df.dropna(subset=["_ts"])

print("rows:", len(df), "pos_rate:", float(df["Is Laundering"].mean()))
print("columns:", df.columns.tolist())

rows: 31898218 pos_rate: 0.0011044504353776574
columns: ['_ts', 'Is Laundering', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Paid', 'Amount Received', 'Receiving Currency', 'Payment Currency', 'Payment Format', 'Log Amount Received', 'tx_hour', 'tx_dow', 'tx_is_weekend', 'tx_hour_sin', 'tx_hour_cos']


In [ ]:
order = np.argsort(df["_ts"].to_numpy())
df = df.iloc[order].reset_index(drop=True)
df["row_id"] = np.arange(len(df), dtype=np.int64)
print("time:", df["_ts"].min(), "->", df["_ts"].max())

time: 2022-09-01 00:00:00 -> 2022-09-28 15:58:00


In [ ]:
N = len(df)
n_train = int(0.70 * N)
n_val   = int(0.15 * N)

tr_idx_np   = np.arange(0, n_train, dtype=np.int64)
val_idx_np  = np.arange(n_train, n_train+n_val, dtype=np.int64)
test_idx_np = np.arange(n_train+n_val, N, dtype=np.int64)

print("Split sizes:", len(tr_idx_np), len(val_idx_np), len(test_idx_np))
print("Pos rates:",
      float(df["Is Laundering"].to_numpy()[tr_idx_np].mean()),
      float(df["Is Laundering"].to_numpy()[val_idx_np].mean()),
      float(df["Is Laundering"].to_numpy()[test_idx_np].mean()))

Split sizes: 22328752 4784732 4784734
Pos rates: 0.0009442085865885019 0.0011467726435512304 0.0018099229782819748


In [ ]:
MAX_EDGES = 3_000_000  # adjust upward later if stable

if len(df) > MAX_EDGES:
    df_edge = df.iloc[:MAX_EDGES].copy()
else:
    df_edge = df.copy()

print("EdgeMLP edges used:", len(df_edge), "of", len(df))

EdgeMLP edges used: 3000000 of 31898218


In [ ]:
src_df = df_edge[["From Bank","Account"]]
dst_df = df_edge[["To Bank","Account.1"]].rename(columns={"To Bank":"From Bank","Account.1":"Account"})
all_df = pd.concat([src_df, dst_df], ignore_index=True)

codes, uniques = pd.factorize(list(map(tuple, all_df.to_numpy())))
m = len(df_edge)

src = codes[:m].astype(np.int64)
dst = codes[m:].astype(np.int64)
y   = df_edge["Is Laundering"].to_numpy(dtype=np.float32)

num_nodes = len(uniques)
print("nodes:", num_nodes, "edges:", m, "pos_rate:", float(y.mean()))

/tmp/ipython-input-2499473087.py:5: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  codes, uniques = pd.factorize(list(map(tuple, all_df.to_numpy())))


nodes: 1545181 edges: 3000000 pos_rate: 0.00019233333296142519


In [ ]:
class EdgeMLP(nn.Module):
    def __init__(self, n_nodes, d=128, dropout=0.2):
        super().__init__()
        self.emb = nn.Embedding(n_nodes, d)
        self.drop = nn.Dropout(dropout)
        self.net = nn.Sequential(
            nn.Linear(4*d, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, s, t):
        Hs = self.drop(self.emb(s))
        Hd = self.drop(self.emb(t))
        x = torch.cat([Hs, Hd, (Hs - Hd).abs(), Hs * Hd], dim=1)
        return self.net(x).squeeze(-1)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

src_gpu = torch.from_numpy(src).long().to(device)
dst_gpu = torch.from_numpy(dst).long().to(device)
y_gpu   = torch.from_numpy(y).float().to(device)

BATCH  = 262_144
EPOCHS = 5  # start small on CPU; raise later if stable
N_FOLDS = 5

def fit_edgemlp(train_idx_np, epochs=EPOCHS):
    model = EdgeMLP(num_nodes, d=128, dropout=0.2).to(device)

    pos_rate = float(y[train_idx_np].mean())
    pos_w = (1.0 - pos_rate) / max(pos_rate, 1e-12)
    pos_weight = torch.tensor([min(pos_w, 1000.0)], device=device)  # cap helps stability
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

    for ep in range(epochs):
        model.train()
        order = train_idx_np.copy()
        np.random.shuffle(order)

        running = 0.0
        for i in range(0, len(order), BATCH):
            b_np = order[i:i+BATCH]
            b = torch.from_numpy(b_np).to(device)

            opt.zero_grad(set_to_none=True)
            logits = model(src_gpu[b], dst_gpu[b])
            loss = loss_fn(logits, y_gpu[b])
            loss.backward()
            opt.step()

            running += float(loss.detach()) * len(b_np)

        print(f"EdgeMLP Epoch {ep:02d} | loss={running/len(order):.6f}")

    return model

@torch.no_grad()
def predict_edgemlp(model, idx_np):
    model.eval()
    out = np.empty(len(idx_np), dtype=np.float32)
    for i in range(0, len(idx_np), BATCH):
        b_np = idx_np[i:i+BATCH]
        b = torch.from_numpy(b_np).to(device)
        out[i:i+len(b_np)] = torch.sigmoid(model(src_gpu[b], dst_gpu[b])).cpu().numpy().astype(np.float32)
    return out

device: cuda


In [ ]:
N_edge = len(df_edge)

# Time split indices for df_edge
n_train_edge = min(int(0.70 * len(df)), N_edge)  # align with original split boundary
n_val_edge   = min(int(0.15 * len(df)), max(0, N_edge - n_train_edge))

tr_edge = np.arange(0, n_train_edge, dtype=np.int64)
va_edge = np.arange(n_train_edge, n_train_edge + n_val_edge, dtype=np.int64)
te_edge = np.arange(n_train_edge + n_val_edge, N_edge, dtype=np.int64)

print("EdgeMLP split sizes:", len(tr_edge), len(va_edge), len(te_edge))
print("EdgeMLP pos rates:", float(y[tr_edge].mean()), float(y[va_edge].mean()) if len(va_edge)>0 else None)

edge_score_edge = np.full(N_edge, np.nan, dtype=np.float32)

# Contiguous time blocks within TRAIN for OOF
blocks = np.array_split(tr_edge, N_FOLDS)

print("OOF EdgeMLP scores on TRAIN (time-respecting blocks)...")
for k in range(N_FOLDS):
    holdout = blocks[k]

    # SAFE FIX: skip first block (no earlier data)
    if k == 0:
        print(f"Block {k+1}/{N_FOLDS}: skipped (no earlier data).")
        continue

    train_f = np.concatenate(blocks[:k])  # now guaranteed non-empty

    m_k = fit_edgemlp(train_f, epochs=EPOCHS)
    edge_score_edge[holdout] = predict_edgemlp(m_k, holdout)

    print(f"Block {k+1}/{N_FOLDS} done.")

print("Final EdgeMLP on full TRAIN → score VAL/TEST + fill skipped TRAIN block if any...")
m_final = fit_edgemlp(tr_edge, epochs=EPOCHS)

# Fill any skipped earliest block (optional; see note below)
nan_tr = tr_edge[np.isnan(edge_score_edge[tr_edge])]
if len(nan_tr) > 0:
    edge_score_edge[nan_tr] = predict_edgemlp(m_final, nan_tr)

if len(va_edge) > 0:
    edge_score_edge[va_edge] = predict_edgemlp(m_final, va_edge)
if len(te_edge) > 0:
    edge_score_edge[te_edge] = predict_edgemlp(m_final, te_edge)

print("NaNs remaining (TRAIN/VAL/TEST):",
      np.isnan(edge_score_edge[tr_edge]).sum(),
      np.isnan(edge_score_edge[va_edge]).sum() if len(va_edge)>0 else 0,
      np.isnan(edge_score_edge[te_edge]).sum() if len(te_edge)>0 else 0)

# Attach edge scores back to df_edge and then map into df (full) by row order
df_edge["edge_score"] = edge_score_edge

EdgeMLP split sizes: 3000000 0 0
EdgeMLP pos rates: 0.00019233333296142519 None
OOF EdgeMLP scores on TRAIN (time-respecting blocks)...
Block 1/5: skipped (no earlier data).
EdgeMLP Epoch 00 | loss=0.612038
EdgeMLP Epoch 01 | loss=0.221496
EdgeMLP Epoch 02 | loss=0.129201
EdgeMLP Epoch 03 | loss=0.125755
EdgeMLP Epoch 04 | loss=0.105478
Block 2/5 done.
EdgeMLP Epoch 00 | loss=0.439688
EdgeMLP Epoch 01 | loss=0.138533
EdgeMLP Epoch 02 | loss=0.124194
EdgeMLP Epoch 03 | loss=0.089224
EdgeMLP Epoch 04 | loss=0.083548
Block 3/5 done.
EdgeMLP Epoch 00 | loss=0.415043
EdgeMLP Epoch 01 | loss=0.272671
EdgeMLP Epoch 02 | loss=0.245201
EdgeMLP Epoch 03 | loss=0.230036
EdgeMLP Epoch 04 | loss=0.224975
Block 4/5 done.
EdgeMLP Epoch 00 | loss=0.485078
EdgeMLP Epoch 01 | loss=0.350306
EdgeMLP Epoch 02 | loss=0.337430
EdgeMLP Epoch 03 | loss=0.327325
EdgeMLP Epoch 04 | loss=0.315883
Block 5/5 done.
Final EdgeMLP on full TRAIN → score VAL/TEST + fill skipped TRAIN block if any...
EdgeMLP Epoch 00 | l

In [ ]:
df["edge_score"] = np.nan

# Because df_edge is the first MAX_EDGES rows of df (time-sorted), row alignment is direct:
df.loc[:len(df_edge)-1, "edge_score"] = df_edge["edge_score"].to_numpy()

print("edge_score filled rows:", df["edge_score"].notna().sum(), "out of", len(df))

edge_score filled rows: 3000000 out of 31898218


In [ ]:
df["Amount Paid"] = pd.to_numeric(df["Amount Paid"], errors="coerce").fillna(0.0)
df["Amount Received"] = pd.to_numeric(df["Amount Received"], errors="coerce").fillna(0.0)

# Use your Log Amount Received if present
if "Log Amount Received" in df.columns:
    df["log_amt_recv"] = pd.to_numeric(df["Log Amount Received"], errors="coerce")
else:
    df["log_amt_recv"] = np.log1p(df["Amount Received"].astype(np.float32))

df["log_amt_paid"] = np.log1p(df["Amount Paid"].astype(np.float32))
df["cross_bank"] = (df["From Bank"].astype(str) != df["To Bank"].astype(str)).astype(np.int8)

# Minimal CatBoost feature set (edit as needed)
cat_cols = ["From Bank", "To Bank", "Receiving Currency", "Payment Currency", "Payment Format"]
num_cols = ["edge_score", "log_amt_paid", "log_amt_recv", "cross_bank",
            "tx_hour", "tx_dow", "tx_is_weekend", "tx_hour_sin", "tx_hour_cos"]

feature_cols = cat_cols + num_cols

# Drop rows where edge_score is missing (rows beyond MAX_EDGES or skipped fold)
df_cb = df.dropna(subset=["edge_score"]).copy()

X = df_cb[feature_cols].copy()
y_cb = df_cb["Is Laundering"].astype(int).to_numpy(np.int32)

# Recompute splits in df_cb by time order position (still time-respecting, but on the filtered df_cb)
n = len(df_cb)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)

Xtr, ytr = X.iloc[:n_train], y_cb[:n_train]
Xva, yva = X.iloc[n_train:n_train+n_val], y_cb[n_train:n_train+n_val]
Xte, yte = X.iloc[n_train+n_val:], y_cb[n_train+n_val:]

print("CatBoost sizes:", Xtr.shape, Xva.shape, Xte.shape)
print("CatBoost pos rates:", ytr.mean(), yva.mean(), yte.mean())

CatBoost sizes: (2100000, 14) (450000, 14) (450000, 14)
CatBoost pos rates: 0.00013904761904761905 0.0003111111111111111 0.0003222222222222222


In [ ]:
cat_idx = [Xtr.columns.get_loc(c) for c in cat_cols]

cb = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.05,
    depth=8,
    loss_function="Logloss",
    eval_metric="PRAUC",
    random_seed=0,
    verbose=200,
    auto_class_weights="Balanced",
    task_type="CPU"
)

cb.fit(Xtr, ytr, eval_set=(Xva, yva), cat_features=cat_idx, use_best_model=True)

pva = cb.predict_proba(Xva)[:, 1]
pte = cb.predict_proba(Xte)[:, 1]

print("\n=== EdgeMLP → CatBoost (time split, leak-free stacking) ===")
print("VAL  ROC:", roc_auc_score(yva, pva))
print("VAL  PR :", average_precision_score(yva, pva))
print("TEST ROC:", roc_auc_score(yte, pte))
print("TEST PR :", average_precision_score(yte, pte))

0:	learn: 0.8512721	test: 0.8871148	best: 0.8871148 (0)	total: 496ms	remaining: 24m 47s
200:	learn: 0.9897905	test: 0.9660870	best: 0.9682393 (117)	total: 2m 27s	remaining: 34m 15s
400:	learn: 0.9983500	test: 0.9560437	best: 0.9682393 (117)	total: 5m 52s	remaining: 38m 7s
600:	learn: 0.9999147	test: 0.9480294	best: 0.9682393 (117)	total: 9m 57s	remaining: 39m 43s
800:	learn: 0.9999917	test: 0.9464659	best: 0.9682393 (117)	total: 14m 6s	remaining: 38m 42s
1000:	learn: 0.9999981	test: 0.9440864	best: 0.9682393 (117)	total: 18m 18s	remaining: 36m 34s
1200:	learn: 0.9999987	test: 0.9446359	best: 0.9682393 (117)	total: 21m 24s	remaining: 32m 4s
1400:	learn: 0.9999987	test: 0.9445804	best: 0.9682393 (117)	total: 23m 58s	remaining: 27m 21s
1600:	learn: 0.9999987	test: 0.9445289	best: 0.9682393 (117)	total: 26m 57s	remaining: 23m 33s
1800:	learn: 0.9999988	test: 0.9443262	best: 0.9682393 (117)	total: 29m 48s	remaining: 19m 50s
2000:	learn: 0.9999987	test: 0.9440496	best: 0.9682393 (117)	total:

In [ ]:
DATASET_NAME = "HI-Medium"          # <-- change each run
MODEL_NAME   = "EdgeMLP+CatBoost"
SPLIT_NAME   = "time_70_15_15"

val_roc = roc_auc_score(yva, pva)
test_roc = roc_auc_score(yte, pte)

val_pr = average_precision_score(yva, pva)
test_pr = average_precision_score(yte, pte)

val_base = float(yva.mean())
test_base = float(yte.mean())

row = {
    "dataset": DATASET_NAME,
    "model": MODEL_NAME,
    "split": SPLIT_NAME,
    "val_roc": val_roc,
    "test_roc": test_roc,
    "val_pr_auc": val_pr,
    "test_pr_auc": test_pr,
    "val_base_rate": val_base,
    "test_base_rate": test_base,
    "val_pr_lift": val_pr / val_base if val_base > 0 else None,
    "test_pr_lift": test_pr / test_base if test_base > 0 else None,
    "n_val": int(len(yva)),
    "n_test": int(len(yte)),
}

new_df = pd.DataFrame([row])

path = Path("aml_model_results.csv")
if path.exists():
    old = pd.read_csv(path)
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(path, index=False)
print(f"Saved → {path} (rows={len(out)})")
out.tail(10)

Saved → aml_model_results.csv (rows=2)


,dataset,model,split,val_roc,test_roc,val_pr_auc,test_pr_auc,val_base_rate,test_base_rate,val_pr_lift,test_pr_lift,n_val,n_test
0,LI-Medium,EdgeMLP+CatBoost,time_70_15_15,0.945931,0.94683,0.003395,0.003856,0.000251,0.000282,13.520729,13.663972,450000,450000
1,HI-Medium,EdgeMLP+CatBoost,time_70_15_15,0.949850,0.93435,0.015138,0.010575,0.000311,0.000322,48.656343,32.819543,450000,450000


## Next, we will be using the LI-Small dataset

In [ ]:
kagglehub.dataset_download("ealtman2019/ibm-transactions-for-anti-money-laundering-aml", path="LI-Small_Trans.csv")

Using Colab cache for faster access to the 'ibm-transactions-for-anti-money-laundering-aml' dataset.


'/kaggle/input/ibm-transactions-for-anti-money-laundering-aml/LI-Small_Trans.csv'

In [ ]:
BASE = "/kaggle/input/ibm-transactions-for-anti-money-laundering-aml"
LI_Strans = pd.read_csv(f"{BASE}/LI-Small_Trans.csv")
LI_Strans.head(5)

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:08,11,8000ECA90,11,8000ECA90,3195403.00,US Dollar,3195403.00,US Dollar,Reinvestment,0
1,2022/09/01 00:21,3402,80021DAD0,3402,80021DAD0,1858.96,US Dollar,1858.96,US Dollar,Reinvestment,0
2,2022/09/01 00:00,11,8000ECA90,1120,8006AA910,592571.00,US Dollar,592571.00,US Dollar,Cheque,0
3,2022/09/01 00:16,3814,8006AD080,3814,8006AD080,12.32,US Dollar,12.32,US Dollar,Reinvestment,0
4,2022/09/01 00:00,20,8006AD530,20,8006AD530,2941.56,US Dollar,2941.56,US Dollar,Reinvestment,0


In [ ]:
LI_Strans["Is Laundering"].unique() #checking the cleanliness of the "Is Laundering" column; all good

array([0, 1])

In [ ]:
null_summary(LI_Strans) #checking for any null values; all good

,null_count


In [ ]:
duplicate_check(LI_Strans) #checking for any duplicates; 8 duplicates

{'total_rows': 6924049,
 'duplicate_rows': np.int64(8),
 'duplicate_rate': np.float64(1.1553933254949525e-06)}

In [ ]:
#dropping the duplicates
print(f"Shape before removing duplicates: {LI_Strans.shape}")
LI_Strans = LI_Strans.drop_duplicates().reset_index(drop=True)
print(f"New shape after removing duplicates: {LI_Strans.shape}")

Shape before removing duplicates: (6924049, 11)
New shape after removing duplicates: (6924041, 11)


In [ ]:
# Check for invalid account names
account_cols = [c for c in LI_Strans.columns if "account" in c.lower()]

if len(account_cols) == 0:
    print("No account columns found (no columns containing 'account').")
else:
    # Basic validity rules:
    # - not null
    # - not empty/whitespace
    # - only allows letters, digits, underscore, hyphen, dot (customize if needed)
    allowed_pattern = r"^[A-Za-z0-9_.-]+$"

    invalid_summary = {}

    for c in account_cols:
        s = LI_Strans[c].astype("string")

        is_null = s.isna()
        is_empty = s.str.strip().eq("")
        bad_chars = ~s.str.match(allowed_pattern, na=False)

        invalid_mask = is_null | is_empty | bad_chars
        invalid_count = int(invalid_mask.sum())

        invalid_summary[c] = {
            "invalid_count": invalid_count,
            "null_count": int(is_null.sum()),
            "empty_count": int(is_empty.sum()),
            "bad_char_count": int(bad_chars.sum())
        }

        print(f"\n[{c}] invalid rows: {invalid_count}")
        print("  null:", invalid_summary[c]["null_count"])
        print("  empty:", invalid_summary[c]["empty_count"])
        print("  bad_chars:", invalid_summary[c]["bad_char_count"])

        if invalid_count > 0:
            examples = LI_Strans.loc[invalid_mask, c].astype("string").head(10).tolist()
            print("  examples:", examples)

    print("\nChecked account columns:", account_cols)


[Account] invalid rows: 0
  null: 0
  empty: 0
  bad_chars: 0

[Account.1] invalid rows: 0
  null: 0
  empty: 0
  bad_chars: 0

Checked account columns: ['Account', 'Account.1']


In [ ]:
# Check for invalid transaction values

value_cols = [c for c in LI_Strans.columns if any(k in c.lower() for k in ["amount", "value"])]

if len(value_cols) == 0:
    print("No transaction value columns found (no columns containing 'amount' or 'value').")
else:
    for c in value_cols:
        x = pd.to_numeric(LI_Strans[c], errors="coerce")

        is_nan = x.isna()
        is_inf = np.isinf(x.to_numpy(dtype=float, copy=False))
        is_neg = x < 0
        is_zero = x == 0

        invalid_mask = is_nan | is_inf | is_neg

        print(f"\n[{c}]")
        print("  total rows:", len(LI_Strans))
        print("  NaN after numeric coercion:", int(is_nan.sum()))
        print("  inf:", int(is_inf.sum()))
        print("  negative:", int(is_neg.sum()))
        print("  zero:", int(is_zero.sum()))
        print("  invalid (NaN/inf/negative):", int(invalid_mask.sum()))

        if int(invalid_mask.sum()) > 0:
            example_rows = LI_Strans.loc[invalid_mask, [c]].head(10)
            print("  first invalid examples:")
            display(example_rows)

    print("\nChecked value columns:", value_cols)


[Amount Received]
  total rows: 6924041
  NaN after numeric coercion: 0
  inf: 0
  negative: 0
  zero: 0
  invalid (NaN/inf/negative): 0

[Amount Paid]
  total rows: 6924041
  NaN after numeric coercion: 0
  inf: 0
  negative: 0
  zero: 0
  invalid (NaN/inf/negative): 0

Checked value columns: ['Amount Received', 'Amount Paid']


In [ ]:
amt = pd.to_numeric(LI_Strans["Amount Received"], errors="coerce")
LI_Strans["Log Amount Received"] = np.log1p(amt)

In [ ]:
timestamp_summary(LI_Strans)

{'min_time': Timestamp('2022-09-01 00:00:00'),
 'max_time': Timestamp('2022-09-17 15:28:00'),
 'null_timestamps': np.int64(0)}

In [ ]:
# Setup for date data preprocessing

TS_COL = "Timestamp"
LABEL_COL = "Is Laundering"

TS_FORMAT = "%Y/%m/%d %H:%M"

# Use True for Large
USE_STREAMING = False
CHUNKSIZE = 2_000_000

CSV_PATH = "/kaggle/input/ibm-transactions-for-anti-money-laundering-aml/LI-Small_Trans.csv"

def _coerce_label_to_int(series):
    return pd.to_numeric(series.astype(str).str.strip(), errors="coerce").fillna(0).astype(int)


def _time_aggs_from_df(df_in, ts_col=TS_COL, label_col=LABEL_COL, ts_format=TS_FORMAT):
    y = _coerce_label_to_int(df_in[label_col]).to_numpy(dtype=np.int8)

    if ts_format is None:
        ts = pd.to_datetime(df_in[ts_col], errors="coerce")
    else:
        ts = pd.to_datetime(df_in[ts_col], format=ts_format, errors="coerce")

    mask = ts.notna().to_numpy()
    bad_ts = int((~mask).sum())

    ts = ts[mask]
    y = y[mask]

    hour = ts.dt.hour.to_numpy(dtype=np.int16)
    dow = ts.dt.dayofweek.to_numpy(dtype=np.int16)  # Mon=0..Sun=6

    hour_total = np.bincount(hour, minlength=24).astype(np.int64)
    hour_pos   = np.bincount(hour, weights=y, minlength=24).astype(np.int64)

    dow_total = np.bincount(dow, minlength=7).astype(np.int64)
    dow_pos   = np.bincount(dow, weights=y, minlength=7).astype(np.int64)

    idx = dow * 24 + hour
    hd_total = np.bincount(idx, minlength=7*24).reshape(7, 24).astype(np.int64)
    hd_pos   = np.bincount(idx, weights=y, minlength=7*24).reshape(7, 24).astype(np.int64)

    return {
        "hour_total": hour_total, "hour_pos": hour_pos,
        "dow_total": dow_total,   "dow_pos": dow_pos,
        "hd_total": hd_total,     "hd_pos": hd_pos,
        "total_rows": int(mask.sum()),
        "bad_ts": bad_ts
    }


def _time_aggs_streaming(csv_path, chunksize=CHUNKSIZE, ts_col=TS_COL, label_col=LABEL_COL, ts_format=TS_FORMAT):
    hour_total = np.zeros(24, dtype=np.int64)
    hour_pos   = np.zeros(24, dtype=np.int64)

    dow_total  = np.zeros(7, dtype=np.int64)
    dow_pos    = np.zeros(7, dtype=np.int64)

    hd_total   = np.zeros((7, 24), dtype=np.int64)
    hd_pos     = np.zeros((7, 24), dtype=np.int64)

    total_rows = 0
    bad_ts = 0

    usecols = [ts_col, label_col]

    for chunk in pd.read_csv(csv_path, usecols=usecols, chunksize=chunksize):
        total_rows += len(chunk)

        y = _coerce_label_to_int(chunk[label_col]).to_numpy(dtype=np.int8)

        if ts_format is None:
            ts = pd.to_datetime(chunk[ts_col], errors="coerce")
        else:
            ts = pd.to_datetime(chunk[ts_col], format=ts_format, errors="coerce")

        mask = ts.notna().to_numpy()
        if not mask.all():
            bad_ts += int((~mask).sum())

        ts = ts[mask]
        y  = y[mask]

        hour = ts.dt.hour.to_numpy(dtype=np.int16)
        dow  = ts.dt.dayofweek.to_numpy(dtype=np.int16)

        hour_total += np.bincount(hour, minlength=24)
        hour_pos   += np.bincount(hour, weights=y, minlength=24).astype(np.int64)

        dow_total += np.bincount(dow, minlength=7)
        dow_pos   += np.bincount(dow, weights=y, minlength=7).astype(np.int64)

        idx = dow * 24 + hour
        flat_total = np.bincount(idx, minlength=7*24).reshape(7, 24)
        flat_pos   = np.bincount(idx, weights=y, minlength=7*24).reshape(7, 24).astype(np.int64)

        hd_total += flat_total
        hd_pos   += flat_pos

    return {
        "hour_total": hour_total, "hour_pos": hour_pos,
        "dow_total": dow_total,   "dow_pos": dow_pos,
        "hd_total": hd_total,     "hd_pos": hd_pos,
        "total_rows": int(total_rows),
        "bad_ts": int(bad_ts)
    }

In [ ]:
if USE_STREAMING:
    agg = _time_aggs_streaming(CSV_PATH, chunksize=CHUNKSIZE, ts_format=TS_FORMAT)
else:
    agg = _time_aggs_from_df(LI_Strans, ts_format=TS_FORMAT)

hour_total = agg["hour_total"]
hour_pos   = agg["hour_pos"]
dow_total  = agg["dow_total"]
dow_pos    = agg["dow_pos"]
hd_total   = agg["hd_total"]
hd_pos     = agg["hd_pos"]

print("Temporal EDA for:", "LI-Small_Trans.csv")
print("Rows used (valid timestamps):", agg["total_rows"])
print("Bad/unparsed timestamps:", agg["bad_ts"])
print("Overall laundering rate:", float(hour_pos.sum() / max(hour_total.sum(), 1)))

Temporal EDA for: LI-Small_Trans.csv
Rows used (valid timestamps): 6924041
Bad/unparsed timestamps: 0
Overall laundering rate: 0.0005148727455542219


In [ ]:
hour_df = pd.DataFrame({
    "hour": np.arange(24),
    "tx_count": hour_total,
    "laundering_count": hour_pos,
    "laundering_rate": hour_pos / np.maximum(hour_total, 1)
})

dow_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
dow_df = pd.DataFrame({
    "dow": np.arange(7),
    "day": dow_names,
    "tx_count": dow_total,
    "laundering_count": dow_pos,
    "laundering_rate": dow_pos / np.maximum(dow_total, 1)
})

hd_rate = hd_pos / np.maximum(hd_total, 1)

In [ ]:
if TS_FORMAT is None:
    ts = pd.to_datetime(LI_Strans[TS_COL], errors="coerce")
else:
    ts = pd.to_datetime(LI_Strans[TS_COL], format=TS_FORMAT, errors="coerce")

LI_Strans["_ts"] = ts
LI_Strans["tx_hour"] = LI_Strans["_ts"].dt.hour.astype("Int16")
LI_Strans["tx_dow"] = LI_Strans["_ts"].dt.dayofweek.astype("Int16")     # Mon=0..Sun=6
LI_Strans["tx_month"] = LI_Strans["_ts"].dt.month.astype("Int16")
LI_Strans["tx_day"] = LI_Strans["_ts"].dt.day.astype("Int16")
LI_Strans["tx_date"] = LI_Strans["_ts"].dt.date                       # python date
LI_Strans["tx_is_weekend"] = LI_Strans["tx_dow"].isin([5, 6]).astype("Int8")
LI_Strans["tx_hour_sin"] = np.sin(2 * np.pi * LI_Strans["tx_hour"].fillna(0) / 24.0)
LI_Strans["tx_hour_cos"] = np.cos(2 * np.pi * LI_Strans["tx_hour"].fillna(0) / 24.0)

In [ ]:
LI_Strans.head()

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,...,Log Amount Received,_ts,tx_hour,tx_dow,tx_month,tx_day,tx_date,tx_is_weekend,tx_hour_sin,tx_hour_cos
0,2022/09/01 00:08,11,8000ECA90,11,8000ECA90,3195403.00,US Dollar,3195403.00,US Dollar,Reinvestment,...,14.977224,2022-09-01 00:08:00,0,3,9,1,2022-09-01,0,0.0,1.0
1,2022/09/01 00:21,3402,80021DAD0,3402,80021DAD0,1858.96,US Dollar,1858.96,US Dollar,Reinvestment,...,7.528310,2022-09-01 00:21:00,0,3,9,1,2022-09-01,0,0.0,1.0
2,2022/09/01 00:00,11,8000ECA90,1120,8006AA910,592571.00,US Dollar,592571.00,US Dollar,Cheque,...,13.292228,2022-09-01 00:00:00,0,3,9,1,2022-09-01,0,0.0,1.0
3,2022/09/01 00:16,3814,8006AD080,3814,8006AD080,12.32,US Dollar,12.32,US Dollar,Reinvestment,...,2.589267,2022-09-01 00:16:00,0,3,9,1,2022-09-01,0,0.0,1.0
4,2022/09/01 00:00,20,8006AD530,20,8006AD530,2941.56,US Dollar,2941.56,US Dollar,Reinvestment,...,7.987035,2022-09-01 00:00:00,0,3,9,1,2022-09-01,0,0.0,1.0


### Modeling

In [ ]:
df0 = LI_Strans

# Must exist from your preprocessing
assert "_ts" in df0.columns, "Expected LI_Strans['_ts'] from preprocessing."
assert "Is Laundering" in df0.columns, "Missing Is Laundering column."

# Keep only modeling columns (edit if you want more features)
keep_cols = [
    "_ts", "Is Laundering",
    "From Bank", "Account", "To Bank", "Account.1",
    "Amount Paid", "Amount Received",
    "Receiving Currency", "Payment Currency", "Payment Format",
    "Log Amount Received",
    "tx_hour", "tx_dow", "tx_is_weekend", "tx_hour_sin", "tx_hour_cos",
]
keep_cols = [c for c in keep_cols if c in df0.columns]
df = df0[keep_cols]

# Label clean (defensive)
y_ser = pd.to_numeric(df["Is Laundering"].astype(str).str.strip(), errors="coerce")
mask = y_ser.isin([0, 1])
df = df.loc[mask].copy()  # copy AFTER narrowing + filtering
df["Is Laundering"] = y_ser.loc[mask].astype(np.float32)

# Drop invalid timestamps
df = df.dropna(subset=["_ts"])

print("rows:", len(df), "pos_rate:", float(df["Is Laundering"].mean()))
print("columns:", df.columns.tolist())

rows: 6924041 pos_rate: 0.0005148727213963866
columns: ['_ts', 'Is Laundering', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Paid', 'Amount Received', 'Receiving Currency', 'Payment Currency', 'Payment Format', 'Log Amount Received', 'tx_hour', 'tx_dow', 'tx_is_weekend', 'tx_hour_sin', 'tx_hour_cos']


In [ ]:
order = np.argsort(df["_ts"].to_numpy())
df = df.iloc[order].reset_index(drop=True)
df["row_id"] = np.arange(len(df), dtype=np.int64)
print("time:", df["_ts"].min(), "->", df["_ts"].max())

time: 2022-09-01 00:00:00 -> 2022-09-17 15:28:00


In [ ]:
N = len(df)
n_train = int(0.70 * N)
n_val   = int(0.15 * N)

tr_idx_np   = np.arange(0, n_train, dtype=np.int64)
val_idx_np  = np.arange(n_train, n_train+n_val, dtype=np.int64)
test_idx_np = np.arange(n_train+n_val, N, dtype=np.int64)

print("Split sizes:", len(tr_idx_np), len(val_idx_np), len(test_idx_np))
print("Pos rates:",
      float(df["Is Laundering"].to_numpy()[tr_idx_np].mean()),
      float(df["Is Laundering"].to_numpy()[val_idx_np].mean()),
      float(df["Is Laundering"].to_numpy()[test_idx_np].mean()))

Split sizes: 4846828 1038606 1038607
Pos rates: 0.00046030106022953987 0.0005613293033093214 0.000723083910997957


In [ ]:
MAX_EDGES = 3_000_000  # adjust upward later if stable

if len(df) > MAX_EDGES:
    df_edge = df.iloc[:MAX_EDGES].copy()
else:
    df_edge = df.copy()

print("EdgeMLP edges used:", len(df_edge), "of", len(df))

EdgeMLP edges used: 3000000 of 6924041


In [ ]:
src_df = df_edge[["From Bank","Account"]]
dst_df = df_edge[["To Bank","Account.1"]].rename(columns={"To Bank":"From Bank","Account.1":"Account"})
all_df = pd.concat([src_df, dst_df], ignore_index=True)

codes, uniques = pd.factorize(list(map(tuple, all_df.to_numpy())))
m = len(df_edge)

src = codes[:m].astype(np.int64)
dst = codes[m:].astype(np.int64)
y   = df_edge["Is Laundering"].to_numpy(dtype=np.float32)

num_nodes = len(uniques)
print("nodes:", num_nodes, "edges:", m, "pos_rate:", float(y.mean()))

/tmp/ipython-input-2499473087.py:5: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  codes, uniques = pd.factorize(list(map(tuple, all_df.to_numpy())))


nodes: 700711 edges: 3000000 pos_rate: 0.00036466665915213525


In [ ]:
class EdgeMLP(nn.Module):
    def __init__(self, n_nodes, d=128, dropout=0.2):
        super().__init__()
        self.emb = nn.Embedding(n_nodes, d)
        self.drop = nn.Dropout(dropout)
        self.net = nn.Sequential(
            nn.Linear(4*d, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, s, t):
        Hs = self.drop(self.emb(s))
        Hd = self.drop(self.emb(t))
        x = torch.cat([Hs, Hd, (Hs - Hd).abs(), Hs * Hd], dim=1)
        return self.net(x).squeeze(-1)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

src_gpu = torch.from_numpy(src).long().to(device)
dst_gpu = torch.from_numpy(dst).long().to(device)
y_gpu   = torch.from_numpy(y).float().to(device)

BATCH  = 262_144
EPOCHS = 5  # start small on CPU; raise later if stable
N_FOLDS = 5

def fit_edgemlp(train_idx_np, epochs=EPOCHS):
    model = EdgeMLP(num_nodes, d=128, dropout=0.2).to(device)

    pos_rate = float(y[train_idx_np].mean())
    pos_w = (1.0 - pos_rate) / max(pos_rate, 1e-12)
    pos_weight = torch.tensor([min(pos_w, 1000.0)], device=device)  # cap helps stability
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

    for ep in range(epochs):
        model.train()
        order = train_idx_np.copy()
        np.random.shuffle(order)

        running = 0.0
        for i in range(0, len(order), BATCH):
            b_np = order[i:i+BATCH]
            b = torch.from_numpy(b_np).to(device)

            opt.zero_grad(set_to_none=True)
            logits = model(src_gpu[b], dst_gpu[b])
            loss = loss_fn(logits, y_gpu[b])
            loss.backward()
            opt.step()

            running += float(loss.detach()) * len(b_np)

        print(f"EdgeMLP Epoch {ep:02d} | loss={running/len(order):.6f}")

    return model

@torch.no_grad()
def predict_edgemlp(model, idx_np):
    model.eval()
    out = np.empty(len(idx_np), dtype=np.float32)
    for i in range(0, len(idx_np), BATCH):
        b_np = idx_np[i:i+BATCH]
        b = torch.from_numpy(b_np).to(device)
        out[i:i+len(b_np)] = torch.sigmoid(model(src_gpu[b], dst_gpu[b])).cpu().numpy().astype(np.float32)
    return out

device: cuda


In [ ]:
N_edge = len(df_edge)

# Time split indices for df_edge
n_train_edge = min(int(0.70 * len(df)), N_edge)  # align with original split boundary
n_val_edge   = min(int(0.15 * len(df)), max(0, N_edge - n_train_edge))

tr_edge = np.arange(0, n_train_edge, dtype=np.int64)
va_edge = np.arange(n_train_edge, n_train_edge + n_val_edge, dtype=np.int64)
te_edge = np.arange(n_train_edge + n_val_edge, N_edge, dtype=np.int64)

print("EdgeMLP split sizes:", len(tr_edge), len(va_edge), len(te_edge))
print("EdgeMLP pos rates:", float(y[tr_edge].mean()), float(y[va_edge].mean()) if len(va_edge)>0 else None)

edge_score_edge = np.full(N_edge, np.nan, dtype=np.float32)

# Contiguous time blocks within TRAIN for OOF
blocks = np.array_split(tr_edge, N_FOLDS)

print("OOF EdgeMLP scores on TRAIN (time-respecting blocks)...")
for k in range(N_FOLDS):
    holdout = blocks[k]

    # SAFE FIX: skip first block (no earlier data)
    if k == 0:
        print(f"Block {k+1}/{N_FOLDS}: skipped (no earlier data).")
        continue

    train_f = np.concatenate(blocks[:k])  # now guaranteed non-empty

    m_k = fit_edgemlp(train_f, epochs=EPOCHS)
    edge_score_edge[holdout] = predict_edgemlp(m_k, holdout)

    print(f"Block {k+1}/{N_FOLDS} done.")

print("Final EdgeMLP on full TRAIN → score VAL/TEST + fill skipped TRAIN block if any...")
m_final = fit_edgemlp(tr_edge, epochs=EPOCHS)

# Fill any skipped earliest block (optional; see note below)
nan_tr = tr_edge[np.isnan(edge_score_edge[tr_edge])]
if len(nan_tr) > 0:
    edge_score_edge[nan_tr] = predict_edgemlp(m_final, nan_tr)

if len(va_edge) > 0:
    edge_score_edge[va_edge] = predict_edgemlp(m_final, va_edge)
if len(te_edge) > 0:
    edge_score_edge[te_edge] = predict_edgemlp(m_final, te_edge)

print("NaNs remaining (TRAIN/VAL/TEST):",
      np.isnan(edge_score_edge[tr_edge]).sum(),
      np.isnan(edge_score_edge[va_edge]).sum() if len(va_edge)>0 else 0,
      np.isnan(edge_score_edge[te_edge]).sum() if len(te_edge)>0 else 0)

# Attach edge scores back to df_edge and then map into df (full) by row order
df_edge["edge_score"] = edge_score_edge

EdgeMLP split sizes: 3000000 0 0
EdgeMLP pos rates: 0.00036466665915213525 None
OOF EdgeMLP scores on TRAIN (time-respecting blocks)...
Block 1/5: skipped (no earlier data).
EdgeMLP Epoch 00 | loss=0.613411
EdgeMLP Epoch 01 | loss=0.326107
EdgeMLP Epoch 02 | loss=0.275536
EdgeMLP Epoch 03 | loss=0.227792
EdgeMLP Epoch 04 | loss=0.210738
Block 2/5 done.
EdgeMLP Epoch 00 | loss=0.609869
EdgeMLP Epoch 01 | loss=0.431604
EdgeMLP Epoch 02 | loss=0.388497
EdgeMLP Epoch 03 | loss=0.380079
EdgeMLP Epoch 04 | loss=0.363782
Block 3/5 done.
EdgeMLP Epoch 00 | loss=0.638348
EdgeMLP Epoch 01 | loss=0.485501
EdgeMLP Epoch 02 | loss=0.459744
EdgeMLP Epoch 03 | loss=0.446552
EdgeMLP Epoch 04 | loss=0.426557
Block 4/5 done.
EdgeMLP Epoch 00 | loss=0.631693
EdgeMLP Epoch 01 | loss=0.529275
EdgeMLP Epoch 02 | loss=0.511059
EdgeMLP Epoch 03 | loss=0.492416
EdgeMLP Epoch 04 | loss=0.475621
Block 5/5 done.
Final EdgeMLP on full TRAIN → score VAL/TEST + fill skipped TRAIN block if any...
EdgeMLP Epoch 00 | l

In [ ]:
df["edge_score"] = np.nan

# Because df_edge is the first MAX_EDGES rows of df (time-sorted), row alignment is direct:
df.loc[:len(df_edge)-1, "edge_score"] = df_edge["edge_score"].to_numpy()

print("edge_score filled rows:", df["edge_score"].notna().sum(), "out of", len(df))

edge_score filled rows: 3000000 out of 6924041


In [ ]:
df["Amount Paid"] = pd.to_numeric(df["Amount Paid"], errors="coerce").fillna(0.0)
df["Amount Received"] = pd.to_numeric(df["Amount Received"], errors="coerce").fillna(0.0)

# Use your Log Amount Received if present
if "Log Amount Received" in df.columns:
    df["log_amt_recv"] = pd.to_numeric(df["Log Amount Received"], errors="coerce")
else:
    df["log_amt_recv"] = np.log1p(df["Amount Received"].astype(np.float32))

df["log_amt_paid"] = np.log1p(df["Amount Paid"].astype(np.float32))
df["cross_bank"] = (df["From Bank"].astype(str) != df["To Bank"].astype(str)).astype(np.int8)

# Minimal CatBoost feature set (edit as needed)
cat_cols = ["From Bank", "To Bank", "Receiving Currency", "Payment Currency", "Payment Format"]
num_cols = ["edge_score", "log_amt_paid", "log_amt_recv", "cross_bank",
            "tx_hour", "tx_dow", "tx_is_weekend", "tx_hour_sin", "tx_hour_cos"]

feature_cols = cat_cols + num_cols

# Drop rows where edge_score is missing (rows beyond MAX_EDGES or skipped fold)
df_cb = df.dropna(subset=["edge_score"]).copy()

X = df_cb[feature_cols].copy()
y_cb = df_cb["Is Laundering"].astype(int).to_numpy(np.int32)

# Recompute splits in df_cb by time order position (still time-respecting, but on the filtered df_cb)
n = len(df_cb)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)

Xtr, ytr = X.iloc[:n_train], y_cb[:n_train]
Xva, yva = X.iloc[n_train:n_train+n_val], y_cb[n_train:n_train+n_val]
Xte, yte = X.iloc[n_train+n_val:], y_cb[n_train+n_val:]

print("CatBoost sizes:", Xtr.shape, Xva.shape, Xte.shape)
print("CatBoost pos rates:", ytr.mean(), yva.mean(), yte.mean())

CatBoost sizes: (2100000, 14) (450000, 14) (450000, 14)
CatBoost pos rates: 0.00024190476190476192 0.00033777777777777777 0.0009644444444444444


In [ ]:
cat_idx = [Xtr.columns.get_loc(c) for c in cat_cols]

cb = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.05,
    depth=8,
    loss_function="Logloss",
    eval_metric="PRAUC",
    random_seed=0,
    verbose=200,
    auto_class_weights="Balanced",
    task_type="CPU"
)

cb.fit(Xtr, ytr, eval_set=(Xva, yva), cat_features=cat_idx, use_best_model=True)

pva = cb.predict_proba(Xva)[:, 1]
pte = cb.predict_proba(Xte)[:, 1]

print("\n=== EdgeMLP → CatBoost (time split, leak-free stacking) ===")
print("VAL  ROC:", roc_auc_score(yva, pva))
print("VAL  PR :", average_precision_score(yva, pva))
print("TEST ROC:", roc_auc_score(yte, pte))
print("TEST PR :", average_precision_score(yte, pte))

0:	learn: 0.8817664	test: 0.8413120	best: 0.8413120 (0)	total: 1.23s	remaining: 1h 1m 14s
200:	learn: 0.9731334	test: 0.9358805	best: 0.9390650 (145)	total: 2m 50s	remaining: 39m 38s
400:	learn: 0.9957848	test: 0.9214909	best: 0.9390650 (145)	total: 6m 17s	remaining: 40m 45s
600:	learn: 0.9996645	test: 0.9117230	best: 0.9390650 (145)	total: 10m 14s	remaining: 40m 51s
800:	learn: 0.9999534	test: 0.9054332	best: 0.9390650 (145)	total: 14m 20s	remaining: 39m 21s
1000:	learn: 0.9999843	test: 0.9031762	best: 0.9390650 (145)	total: 18m 21s	remaining: 36m 39s
1200:	learn: 0.9999938	test: 0.9010323	best: 0.9390650 (145)	total: 22m 23s	remaining: 33m 31s
1400:	learn: 0.9999959	test: 0.9012805	best: 0.9390650 (145)	total: 26m 20s	remaining: 30m 4s
1600:	learn: 0.9999966	test: 0.9014174	best: 0.9390650 (145)	total: 29m 51s	remaining: 26m 5s
1800:	learn: 0.9999967	test: 0.9015115	best: 0.9390650 (145)	total: 33m 2s	remaining: 21m 59s
2000:	learn: 0.9999969	test: 0.9009926	best: 0.9390650 (145)	tot

In [ ]:
DATASET_NAME = "LI-Small"          # <-- change each run
MODEL_NAME   = "EdgeMLP+CatBoost"
SPLIT_NAME   = "time_70_15_15"

val_roc = roc_auc_score(yva, pva)
test_roc = roc_auc_score(yte, pte)

val_pr = average_precision_score(yva, pva)
test_pr = average_precision_score(yte, pte)

val_base = float(yva.mean())
test_base = float(yte.mean())

row = {
    "dataset": DATASET_NAME,
    "model": MODEL_NAME,
    "split": SPLIT_NAME,
    "val_roc": val_roc,
    "test_roc": test_roc,
    "val_pr_auc": val_pr,
    "test_pr_auc": test_pr,
    "val_base_rate": val_base,
    "test_base_rate": test_base,
    "val_pr_lift": val_pr / val_base if val_base > 0 else None,
    "test_pr_lift": test_pr / test_base if test_base > 0 else None,
    "n_val": int(len(yva)),
    "n_test": int(len(yte)),
}

new_df = pd.DataFrame([row])

path = Path("aml_model_results.csv")
if path.exists():
    old = pd.read_csv(path)
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(path, index=False)
print(f"Saved → {path} (rows={len(out)})")
out.tail(10)

Saved → aml_model_results.csv (rows=3)


,dataset,model,split,val_roc,test_roc,val_pr_auc,test_pr_auc,val_base_rate,test_base_rate,val_pr_lift,test_pr_lift,n_val,n_test
0,LI-Medium,EdgeMLP+CatBoost,time_70_15_15,0.945931,0.94683,0.003395,0.003856,0.000251,0.000282,13.520729,13.663972,450000,450000
1,HI-Medium,EdgeMLP+CatBoost,time_70_15_15,0.949850,0.93435,0.015138,0.010575,0.000311,0.000322,48.656343,32.819543,450000,450000
2,LI-Small,EdgeMLP+CatBoost,time_70_15_15,0.936498,0.93880,0.006706,0.033692,0.000338,0.000964,19.854709,34.933790,450000,450000


## Finally, we will be using the HI-Small dataset

In [4]:
kagglehub.dataset_download("ealtman2019/ibm-transactions-for-anti-money-laundering-aml", path="HI-Small_Trans.csv")

100%|██████████| 454M/454M [00:07<00:00, 62.8MB/s]


'/root/.cache/kagglehub/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/versions/8/HI-Small_Trans.csv'

In [7]:
BASE = "/root/.cache/kagglehub/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/versions/8"
HI_Strans = pd.read_csv(f"{BASE}/HI-Small_Trans.csv")
HI_Strans.head(5)

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


In [8]:
HI_Strans["Is Laundering"].unique() #checking the cleanliness of the "Is Laundering" column; all good

array([0, 1])

In [9]:
null_summary(HI_Strans) #checking for any null values; all good

,null_count


In [10]:
duplicate_check(HI_Strans) #checking for any duplicates; 9 duplicates

{'total_rows': 5078345,
 'duplicate_rows': np.int64(9),
 'duplicate_rate': np.float64(1.7722309138114878e-06)}

In [11]:
#dropping the duplicates
print(f"Shape before removing duplicates: {HI_Strans.shape}")
HI_Strans = HI_Strans.drop_duplicates().reset_index(drop=True)
print(f"New shape after removing duplicates: {HI_Strans.shape}")

Shape before removing duplicates: (5078345, 11)
New shape after removing duplicates: (5078336, 11)


In [12]:
# Check for invalid account names
account_cols = [c for c in HI_Strans.columns if "account" in c.lower()]

if len(account_cols) == 0:
    print("No account columns found (no columns containing 'account').")
else:
    # Basic validity rules:
    # - not null
    # - not empty/whitespace
    # - only allows letters, digits, underscore, hyphen, dot (customize if needed)
    allowed_pattern = r"^[A-Za-z0-9_.-]+$"

    invalid_summary = {}

    for c in account_cols:
        s = HI_Strans[c].astype("string")

        is_null = s.isna()
        is_empty = s.str.strip().eq("")
        bad_chars = ~s.str.match(allowed_pattern, na=False)

        invalid_mask = is_null | is_empty | bad_chars
        invalid_count = int(invalid_mask.sum())

        invalid_summary[c] = {
            "invalid_count": invalid_count,
            "null_count": int(is_null.sum()),
            "empty_count": int(is_empty.sum()),
            "bad_char_count": int(bad_chars.sum())
        }

        print(f"\n[{c}] invalid rows: {invalid_count}")
        print("  null:", invalid_summary[c]["null_count"])
        print("  empty:", invalid_summary[c]["empty_count"])
        print("  bad_chars:", invalid_summary[c]["bad_char_count"])

        if invalid_count > 0:
            examples = HI_Strans.loc[invalid_mask, c].astype("string").head(10).tolist()
            print("  examples:", examples)

    print("\nChecked account columns:", account_cols)


[Account] invalid rows: 0
  null: 0
  empty: 0
  bad_chars: 0

[Account.1] invalid rows: 0
  null: 0
  empty: 0
  bad_chars: 0

Checked account columns: ['Account', 'Account.1']


In [13]:
# Check for invalid transaction values

value_cols = [c for c in HI_Strans.columns if any(k in c.lower() for k in ["amount", "value"])]

if len(value_cols) == 0:
    print("No transaction value columns found (no columns containing 'amount' or 'value').")
else:
    for c in value_cols:
        x = pd.to_numeric(HI_Strans[c], errors="coerce")

        is_nan = x.isna()
        is_inf = np.isinf(x.to_numpy(dtype=float, copy=False))
        is_neg = x < 0
        is_zero = x == 0

        invalid_mask = is_nan | is_inf | is_neg

        print(f"\n[{c}]")
        print("  total rows:", len(HI_Strans))
        print("  NaN after numeric coercion:", int(is_nan.sum()))
        print("  inf:", int(is_inf.sum()))
        print("  negative:", int(is_neg.sum()))
        print("  zero:", int(is_zero.sum()))
        print("  invalid (NaN/inf/negative):", int(invalid_mask.sum()))

        if int(invalid_mask.sum()) > 0:
            example_rows = HI_Strans.loc[invalid_mask, [c]].head(10)
            print("  first invalid examples:")
            display(example_rows)

    print("\nChecked value columns:", value_cols)


[Amount Received]
  total rows: 5078336
  NaN after numeric coercion: 0
  inf: 0
  negative: 0
  zero: 0
  invalid (NaN/inf/negative): 0

[Amount Paid]
  total rows: 5078336
  NaN after numeric coercion: 0
  inf: 0
  negative: 0
  zero: 0
  invalid (NaN/inf/negative): 0

Checked value columns: ['Amount Received', 'Amount Paid']


In [14]:
amt = pd.to_numeric(HI_Strans["Amount Received"], errors="coerce")
HI_Strans["Log Amount Received"] = np.log1p(amt)

In [15]:
timestamp_summary(HI_Strans)

{'min_time': Timestamp('2022-09-01 00:00:00'),
 'max_time': Timestamp('2022-09-18 16:18:00'),
 'null_timestamps': np.int64(0)}

In [16]:
# Setup for date data preprocessing

TS_COL = "Timestamp"
LABEL_COL = "Is Laundering"

TS_FORMAT = "%Y/%m/%d %H:%M"

# Use True for Large
USE_STREAMING = False
CHUNKSIZE = 2_000_000

CSV_PATH = "/root/.cache/kagglehub/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/versions/8/HI-Small_Trans.csv"

def _coerce_label_to_int(series):
    return pd.to_numeric(series.astype(str).str.strip(), errors="coerce").fillna(0).astype(int)


def _time_aggs_from_df(df_in, ts_col=TS_COL, label_col=LABEL_COL, ts_format=TS_FORMAT):
    y = _coerce_label_to_int(df_in[label_col]).to_numpy(dtype=np.int8)

    if ts_format is None:
        ts = pd.to_datetime(df_in[ts_col], errors="coerce")
    else:
        ts = pd.to_datetime(df_in[ts_col], format=ts_format, errors="coerce")

    mask = ts.notna().to_numpy()
    bad_ts = int((~mask).sum())

    ts = ts[mask]
    y = y[mask]

    hour = ts.dt.hour.to_numpy(dtype=np.int16)
    dow = ts.dt.dayofweek.to_numpy(dtype=np.int16)  # Mon=0..Sun=6

    hour_total = np.bincount(hour, minlength=24).astype(np.int64)
    hour_pos   = np.bincount(hour, weights=y, minlength=24).astype(np.int64)

    dow_total = np.bincount(dow, minlength=7).astype(np.int64)
    dow_pos   = np.bincount(dow, weights=y, minlength=7).astype(np.int64)

    idx = dow * 24 + hour
    hd_total = np.bincount(idx, minlength=7*24).reshape(7, 24).astype(np.int64)
    hd_pos   = np.bincount(idx, weights=y, minlength=7*24).reshape(7, 24).astype(np.int64)

    return {
        "hour_total": hour_total, "hour_pos": hour_pos,
        "dow_total": dow_total,   "dow_pos": dow_pos,
        "hd_total": hd_total,     "hd_pos": hd_pos,
        "total_rows": int(mask.sum()),
        "bad_ts": bad_ts
    }


def _time_aggs_streaming(csv_path, chunksize=CHUNKSIZE, ts_col=TS_COL, label_col=LABEL_COL, ts_format=TS_FORMAT):
    hour_total = np.zeros(24, dtype=np.int64)
    hour_pos   = np.zeros(24, dtype=np.int64)

    dow_total  = np.zeros(7, dtype=np.int64)
    dow_pos    = np.zeros(7, dtype=np.int64)

    hd_total   = np.zeros((7, 24), dtype=np.int64)
    hd_pos     = np.zeros((7, 24), dtype=np.int64)

    total_rows = 0
    bad_ts = 0

    usecols = [ts_col, label_col]

    for chunk in pd.read_csv(csv_path, usecols=usecols, chunksize=chunksize):
        total_rows += len(chunk)

        y = _coerce_label_to_int(chunk[label_col]).to_numpy(dtype=np.int8)

        if ts_format is None:
            ts = pd.to_datetime(chunk[ts_col], errors="coerce")
        else:
            ts = pd.to_datetime(chunk[ts_col], format=ts_format, errors="coerce")

        mask = ts.notna().to_numpy()
        if not mask.all():
            bad_ts += int((~mask).sum())

        ts = ts[mask]
        y  = y[mask]

        hour = ts.dt.hour.to_numpy(dtype=np.int16)
        dow  = ts.dt.dayofweek.to_numpy(dtype=np.int16)

        hour_total += np.bincount(hour, minlength=24)
        hour_pos   += np.bincount(hour, weights=y, minlength=24).astype(np.int64)

        dow_total += np.bincount(dow, minlength=7)
        dow_pos   += np.bincount(dow, weights=y, minlength=7).astype(np.int64)

        idx = dow * 24 + hour
        flat_total = np.bincount(idx, minlength=7*24).reshape(7, 24)
        flat_pos   = np.bincount(idx, weights=y, minlength=7*24).reshape(7, 24).astype(np.int64)

        hd_total += flat_total
        hd_pos   += flat_pos

    return {
        "hour_total": hour_total, "hour_pos": hour_pos,
        "dow_total": dow_total,   "dow_pos": dow_pos,
        "hd_total": hd_total,     "hd_pos": hd_pos,
        "total_rows": int(total_rows),
        "bad_ts": int(bad_ts)
    }

In [17]:
if USE_STREAMING:
    agg = _time_aggs_streaming(CSV_PATH, chunksize=CHUNKSIZE, ts_format=TS_FORMAT)
else:
    agg = _time_aggs_from_df(HI_Strans, ts_format=TS_FORMAT)

hour_total = agg["hour_total"]
hour_pos   = agg["hour_pos"]
dow_total  = agg["dow_total"]
dow_pos    = agg["dow_pos"]
hd_total   = agg["hd_total"]
hd_pos     = agg["hd_pos"]

print("Temporal EDA for:", "HI-Small_Trans.csv")
print("Rows used (valid timestamps):", agg["total_rows"])
print("Bad/unparsed timestamps:", agg["bad_ts"])
print("Overall laundering rate:", float(hour_pos.sum() / max(hour_total.sum(), 1)))

Temporal EDA for: HI-Small_Trans.csv
Rows used (valid timestamps): 5078336
Bad/unparsed timestamps: 0
Overall laundering rate: 0.0010194284111961084


In [18]:
hour_df = pd.DataFrame({
    "hour": np.arange(24),
    "tx_count": hour_total,
    "laundering_count": hour_pos,
    "laundering_rate": hour_pos / np.maximum(hour_total, 1)
})

dow_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
dow_df = pd.DataFrame({
    "dow": np.arange(7),
    "day": dow_names,
    "tx_count": dow_total,
    "laundering_count": dow_pos,
    "laundering_rate": dow_pos / np.maximum(dow_total, 1)
})

hd_rate = hd_pos / np.maximum(hd_total, 1)

In [19]:
if TS_FORMAT is None:
    ts = pd.to_datetime(HI_Strans[TS_COL], errors="coerce")
else:
    ts = pd.to_datetime(HI_Strans[TS_COL], format=TS_FORMAT, errors="coerce")

HI_Strans["_ts"] = ts
HI_Strans["tx_hour"] = HI_Strans["_ts"].dt.hour.astype("Int16")
HI_Strans["tx_dow"] = HI_Strans["_ts"].dt.dayofweek.astype("Int16")     # Mon=0..Sun=6
HI_Strans["tx_month"] = HI_Strans["_ts"].dt.month.astype("Int16")
HI_Strans["tx_day"] = HI_Strans["_ts"].dt.day.astype("Int16")
HI_Strans["tx_date"] = HI_Strans["_ts"].dt.date                       # python date
HI_Strans["tx_is_weekend"] = HI_Strans["tx_dow"].isin([5, 6]).astype("Int8")
HI_Strans["tx_hour_sin"] = np.sin(2 * np.pi * HI_Strans["tx_hour"].fillna(0) / 24.0)
HI_Strans["tx_hour_cos"] = np.cos(2 * np.pi * HI_Strans["tx_hour"].fillna(0) / 24.0)

In [20]:
HI_Strans.head()

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,...,Log Amount Received,_ts,tx_hour,tx_dow,tx_month,tx_day,tx_date,tx_is_weekend,tx_hour_sin,tx_hour_cos
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,...,8.215639,2022-09-01 00:20:00,0,3,9,1,2022-09-01,0,0.0,1.0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,...,0.009950,2022-09-01 00:20:00,0,3,9,1,2022-09-01,0,0.0,1.0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,...,9.594008,2022-09-01 00:00:00,0,3,9,1,2022-09-01,0,0.0,1.0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,...,7.940217,2022-09-01 00:02:00,0,3,9,1,2022-09-01,0,0.0,1.0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,...,10.510095,2022-09-01 00:06:00,0,3,9,1,2022-09-01,0,0.0,1.0


### Modeling

In [21]:
df0 = HI_Strans

# Must exist from your preprocessing
assert "_ts" in df0.columns, "Expected HI_Strans['_ts'] from preprocessing."
assert "Is Laundering" in df0.columns, "Missing Is Laundering column."

# Keep only modeling columns (edit if you want more features)
keep_cols = [
    "_ts", "Is Laundering",
    "From Bank", "Account", "To Bank", "Account.1",
    "Amount Paid", "Amount Received",
    "Receiving Currency", "Payment Currency", "Payment Format",
    "Log Amount Received",
    "tx_hour", "tx_dow", "tx_is_weekend", "tx_hour_sin", "tx_hour_cos",
]
keep_cols = [c for c in keep_cols if c in df0.columns]
df = df0[keep_cols]

# Label clean (defensive)
y_ser = pd.to_numeric(df["Is Laundering"].astype(str).str.strip(), errors="coerce")
mask = y_ser.isin([0, 1])
df = df.loc[mask].copy()  # copy AFTER narrowing + filtering
df["Is Laundering"] = y_ser.loc[mask].astype(np.float32)

# Drop invalid timestamps
df = df.dropna(subset=["_ts"])

print("rows:", len(df), "pos_rate:", float(df["Is Laundering"].mean()))
print("columns:", df.columns.tolist())

rows: 5078336 pos_rate: 0.0010194283677265048
columns: ['_ts', 'Is Laundering', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Paid', 'Amount Received', 'Receiving Currency', 'Payment Currency', 'Payment Format', 'Log Amount Received', 'tx_hour', 'tx_dow', 'tx_is_weekend', 'tx_hour_sin', 'tx_hour_cos']


In [22]:
order = np.argsort(df["_ts"].to_numpy())
df = df.iloc[order].reset_index(drop=True)
df["row_id"] = np.arange(len(df), dtype=np.int64)
print("time:", df["_ts"].min(), "->", df["_ts"].max())

time: 2022-09-01 00:00:00 -> 2022-09-18 16:18:00


In [23]:
N = len(df)
n_train = int(0.70 * N)
n_val   = int(0.15 * N)

tr_idx_np   = np.arange(0, n_train, dtype=np.int64)
val_idx_np  = np.arange(n_train, n_train+n_val, dtype=np.int64)
test_idx_np = np.arange(n_train+n_val, N, dtype=np.int64)

print("Split sizes:", len(tr_idx_np), len(val_idx_np), len(test_idx_np))
print("Pos rates:",
      float(df["Is Laundering"].to_numpy()[tr_idx_np].mean()),
      float(df["Is Laundering"].to_numpy()[val_idx_np].mean()),
      float(df["Is Laundering"].to_numpy()[test_idx_np].mean()))

Split sizes: 3554835 761750 761751
Pos rates: 0.0008034128113649786 0.0009977027075365186 0.0020492260809987783


In [24]:
MAX_EDGES = 3_000_000  # adjust upward later if stable

if len(df) > MAX_EDGES:
    df_edge = df.iloc[:MAX_EDGES].copy()
else:
    df_edge = df.copy()

print("EdgeMLP edges used:", len(df_edge), "of", len(df))

EdgeMLP edges used: 3000000 of 5078336


In [25]:
src_df = df_edge[["From Bank","Account"]]
dst_df = df_edge[["To Bank","Account.1"]].rename(columns={"To Bank":"From Bank","Account.1":"Account"})
all_df = pd.concat([src_df, dst_df], ignore_index=True)

codes, uniques = pd.factorize(list(map(tuple, all_df.to_numpy())))
m = len(df_edge)

src = codes[:m].astype(np.int64)
dst = codes[m:].astype(np.int64)
y   = df_edge["Is Laundering"].to_numpy(dtype=np.float32)

num_nodes = len(uniques)
print("nodes:", num_nodes, "edges:", m, "pos_rate:", float(y.mean()))

/tmp/ipython-input-2499473087.py:5: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  codes, uniques = pd.factorize(list(map(tuple, all_df.to_numpy())))


nodes: 512642 edges: 3000000 pos_rate: 0.0007389999809674919


In [26]:
class EdgeMLP(nn.Module):
    def __init__(self, n_nodes, d=128, dropout=0.2):
        super().__init__()
        self.emb = nn.Embedding(n_nodes, d)
        self.drop = nn.Dropout(dropout)
        self.net = nn.Sequential(
            nn.Linear(4*d, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, s, t):
        Hs = self.drop(self.emb(s))
        Hd = self.drop(self.emb(t))
        x = torch.cat([Hs, Hd, (Hs - Hd).abs(), Hs * Hd], dim=1)
        return self.net(x).squeeze(-1)

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

src_gpu = torch.from_numpy(src).long().to(device)
dst_gpu = torch.from_numpy(dst).long().to(device)
y_gpu   = torch.from_numpy(y).float().to(device)

BATCH  = 262_144
EPOCHS = 5  # start small on CPU; raise later if stable
N_FOLDS = 5

def fit_edgemlp(train_idx_np, epochs=EPOCHS):
    model = EdgeMLP(num_nodes, d=128, dropout=0.2).to(device)

    pos_rate = float(y[train_idx_np].mean())
    pos_w = (1.0 - pos_rate) / max(pos_rate, 1e-12)
    pos_weight = torch.tensor([min(pos_w, 1000.0)], device=device)  # cap helps stability
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

    for ep in range(epochs):
        model.train()
        order = train_idx_np.copy()
        np.random.shuffle(order)

        running = 0.0
        for i in range(0, len(order), BATCH):
            b_np = order[i:i+BATCH]
            b = torch.from_numpy(b_np).to(device)

            opt.zero_grad(set_to_none=True)
            logits = model(src_gpu[b], dst_gpu[b])
            loss = loss_fn(logits, y_gpu[b])
            loss.backward()
            opt.step()

            running += float(loss.detach()) * len(b_np)

        print(f"EdgeMLP Epoch {ep:02d} | loss={running/len(order):.6f}")

    return model

@torch.no_grad()
def predict_edgemlp(model, idx_np):
    model.eval()
    out = np.empty(len(idx_np), dtype=np.float32)
    for i in range(0, len(idx_np), BATCH):
        b_np = idx_np[i:i+BATCH]
        b = torch.from_numpy(b_np).to(device)
        out[i:i+len(b_np)] = torch.sigmoid(model(src_gpu[b], dst_gpu[b])).cpu().numpy().astype(np.float32)
    return out

device: cuda


In [28]:
N_edge = len(df_edge)

# Time split indices for df_edge
n_train_edge = min(int(0.70 * len(df)), N_edge)  # align with original split boundary
n_val_edge   = min(int(0.15 * len(df)), max(0, N_edge - n_train_edge))

tr_edge = np.arange(0, n_train_edge, dtype=np.int64)
va_edge = np.arange(n_train_edge, n_train_edge + n_val_edge, dtype=np.int64)
te_edge = np.arange(n_train_edge + n_val_edge, N_edge, dtype=np.int64)

print("EdgeMLP split sizes:", len(tr_edge), len(va_edge), len(te_edge))
print("EdgeMLP pos rates:", float(y[tr_edge].mean()), float(y[va_edge].mean()) if len(va_edge)>0 else None)

edge_score_edge = np.full(N_edge, np.nan, dtype=np.float32)

# Contiguous time blocks within TRAIN for OOF
blocks = np.array_split(tr_edge, N_FOLDS)

print("OOF EdgeMLP scores on TRAIN (time-respecting blocks)...")
for k in range(N_FOLDS):
    holdout = blocks[k]

    # SAFE FIX: skip first block (no earlier data)
    if k == 0:
        print(f"Block {k+1}/{N_FOLDS}: skipped (no earlier data).")
        continue

    train_f = np.concatenate(blocks[:k])  # now guaranteed non-empty

    m_k = fit_edgemlp(train_f, epochs=EPOCHS)
    edge_score_edge[holdout] = predict_edgemlp(m_k, holdout)

    print(f"Block {k+1}/{N_FOLDS} done.")

print("Final EdgeMLP on full TRAIN → score VAL/TEST + fill skipped TRAIN block if any...")
m_final = fit_edgemlp(tr_edge, epochs=EPOCHS)

# Fill any skipped earliest block (optional; see note below)
nan_tr = tr_edge[np.isnan(edge_score_edge[tr_edge])]
if len(nan_tr) > 0:
    edge_score_edge[nan_tr] = predict_edgemlp(m_final, nan_tr)

if len(va_edge) > 0:
    edge_score_edge[va_edge] = predict_edgemlp(m_final, va_edge)
if len(te_edge) > 0:
    edge_score_edge[te_edge] = predict_edgemlp(m_final, te_edge)

print("NaNs remaining (TRAIN/VAL/TEST):",
      np.isnan(edge_score_edge[tr_edge]).sum(),
      np.isnan(edge_score_edge[va_edge]).sum() if len(va_edge)>0 else 0,
      np.isnan(edge_score_edge[te_edge]).sum() if len(te_edge)>0 else 0)

# Attach edge scores back to df_edge and then map into df (full) by row order
df_edge["edge_score"] = edge_score_edge

EdgeMLP split sizes: 3000000 0 0
EdgeMLP pos rates: 0.0007389999809674919 None
OOF EdgeMLP scores on TRAIN (time-respecting blocks)...
Block 1/5: skipped (no earlier data).
EdgeMLP Epoch 00 | loss=0.728284
EdgeMLP Epoch 01 | loss=0.491146
EdgeMLP Epoch 02 | loss=0.426078
EdgeMLP Epoch 03 | loss=0.372470
EdgeMLP Epoch 04 | loss=0.358080
Block 2/5 done.
EdgeMLP Epoch 00 | loss=0.736753
EdgeMLP Epoch 01 | loss=0.583351
EdgeMLP Epoch 02 | loss=0.552218
EdgeMLP Epoch 03 | loss=0.535574
EdgeMLP Epoch 04 | loss=0.518070
Block 3/5 done.
EdgeMLP Epoch 00 | loss=0.825008
EdgeMLP Epoch 01 | loss=0.724024
EdgeMLP Epoch 02 | loss=0.693367
EdgeMLP Epoch 03 | loss=0.673950
EdgeMLP Epoch 04 | loss=0.650607
Block 4/5 done.
EdgeMLP Epoch 00 | loss=1.064362
EdgeMLP Epoch 01 | loss=0.990809
EdgeMLP Epoch 02 | loss=0.968178
EdgeMLP Epoch 03 | loss=0.934587
EdgeMLP Epoch 04 | loss=0.906682
Block 5/5 done.
Final EdgeMLP on full TRAIN → score VAL/TEST + fill skipped TRAIN block if any...
EdgeMLP Epoch 00 | lo

In [29]:
df["edge_score"] = np.nan

# Because df_edge is the first MAX_EDGES rows of df (time-sorted), row alignment is direct:
df.loc[:len(df_edge)-1, "edge_score"] = df_edge["edge_score"].to_numpy()

print("edge_score filled rows:", df["edge_score"].notna().sum(), "out of", len(df))

edge_score filled rows: 3000000 out of 5078336


In [30]:
df["Amount Paid"] = pd.to_numeric(df["Amount Paid"], errors="coerce").fillna(0.0)
df["Amount Received"] = pd.to_numeric(df["Amount Received"], errors="coerce").fillna(0.0)

# Use your Log Amount Received if present
if "Log Amount Received" in df.columns:
    df["log_amt_recv"] = pd.to_numeric(df["Log Amount Received"], errors="coerce")
else:
    df["log_amt_recv"] = np.log1p(df["Amount Received"].astype(np.float32))

df["log_amt_paid"] = np.log1p(df["Amount Paid"].astype(np.float32))
df["cross_bank"] = (df["From Bank"].astype(str) != df["To Bank"].astype(str)).astype(np.int8)

# Minimal CatBoost feature set (edit as needed)
cat_cols = ["From Bank", "To Bank", "Receiving Currency", "Payment Currency", "Payment Format"]
num_cols = ["edge_score", "log_amt_paid", "log_amt_recv", "cross_bank",
            "tx_hour", "tx_dow", "tx_is_weekend", "tx_hour_sin", "tx_hour_cos"]

feature_cols = cat_cols + num_cols

# Drop rows where edge_score is missing (rows beyond MAX_EDGES or skipped fold)
df_cb = df.dropna(subset=["edge_score"]).copy()

X = df_cb[feature_cols].copy()
y_cb = df_cb["Is Laundering"].astype(int).to_numpy(np.int32)

# Recompute splits in df_cb by time order position (still time-respecting, but on the filtered df_cb)
n = len(df_cb)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)

Xtr, ytr = X.iloc[:n_train], y_cb[:n_train]
Xva, yva = X.iloc[n_train:n_train+n_val], y_cb[n_train:n_train+n_val]
Xte, yte = X.iloc[n_train+n_val:], y_cb[n_train+n_val:]

print("CatBoost sizes:", Xtr.shape, Xva.shape, Xte.shape)
print("CatBoost pos rates:", ytr.mean(), yva.mean(), yte.mean())

CatBoost sizes: (2100000, 14) (450000, 14) (450000, 14)
CatBoost pos rates: 0.0005447619047619048 0.0013688888888888889 0.0010155555555555556


In [31]:
cat_idx = [Xtr.columns.get_loc(c) for c in cat_cols]

cb = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.05,
    depth=8,
    loss_function="Logloss",
    eval_metric="PRAUC",
    random_seed=0,
    verbose=200,
    auto_class_weights="Balanced",
    task_type="CPU"
)

cb.fit(Xtr, ytr, eval_set=(Xva, yva), cat_features=cat_idx, use_best_model=True)

pva = cb.predict_proba(Xva)[:, 1]
pte = cb.predict_proba(Xte)[:, 1]

print("\n=== EdgeMLP → CatBoost (time split, leak-free stacking) ===")
print("VAL  ROC:", roc_auc_score(yva, pva))
print("VAL  PR :", average_precision_score(yva, pva))
print("TEST ROC:", roc_auc_score(yte, pte))
print("TEST PR :", average_precision_score(yte, pte))

0:	learn: 0.8813491	test: 0.9503185	best: 0.9503185 (0)	total: 490ms	remaining: 24m 29s
200:	learn: 0.9797742	test: 0.9852808	best: 0.9853950 (191)	total: 2m 54s	remaining: 40m 31s
400:	learn: 0.9908589	test: 0.9848266	best: 0.9856968 (322)	total: 6m 17s	remaining: 40m 48s
600:	learn: 0.9964504	test: 0.9834634	best: 0.9856968 (322)	total: 10m 9s	remaining: 40m 34s
800:	learn: 0.9984736	test: 0.9830150	best: 0.9856968 (322)	total: 14m 2s	remaining: 38m 31s
1000:	learn: 0.9991759	test: 0.9822034	best: 0.9856968 (322)	total: 17m 58s	remaining: 35m 54s
1200:	learn: 0.9998360	test: 0.9814109	best: 0.9856968 (322)	total: 21m 55s	remaining: 32m 50s
1400:	learn: 0.9999154	test: 0.9809973	best: 0.9856968 (322)	total: 25m 58s	remaining: 29m 38s
1600:	learn: 0.9999504	test: 0.9802102	best: 0.9856968 (322)	total: 29m 59s	remaining: 26m 12s
1800:	learn: 0.9999669	test: 0.9798349	best: 0.9856968 (322)	total: 33m 49s	remaining: 22m 31s
2000:	learn: 0.9999731	test: 0.9797133	best: 0.9856968 (322)	tota

In [32]:
DATASET_NAME = "HI-Small"          # <-- change each run
MODEL_NAME   = "EdgeMLP+CatBoost"
SPLIT_NAME   = "time_70_15_15"

val_roc = roc_auc_score(yva, pva)
test_roc = roc_auc_score(yte, pte)

val_pr = average_precision_score(yva, pva)
test_pr = average_precision_score(yte, pte)

val_base = float(yva.mean())
test_base = float(yte.mean())

row = {
    "dataset": DATASET_NAME,
    "model": MODEL_NAME,
    "split": SPLIT_NAME,
    "val_roc": val_roc,
    "test_roc": test_roc,
    "val_pr_auc": val_pr,
    "test_pr_auc": test_pr,
    "val_base_rate": val_base,
    "test_base_rate": test_base,
    "val_pr_lift": val_pr / val_base if val_base > 0 else None,
    "test_pr_lift": test_pr / test_base if test_base > 0 else None,
    "n_val": int(len(yva)),
    "n_test": int(len(yte)),
}

new_df = pd.DataFrame([row])

path = Path("aml_model_results.csv")
if path.exists():
    old = pd.read_csv(path)
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(path, index=False)
print(f"Saved → {path} (rows={len(out)})")
out.tail(10)

Saved → aml_model_results.csv (rows=1)


,dataset,model,split,val_roc,test_roc,...,test_base_rate,val_pr_lift,test_pr_lift,n_val,n_test
0,HI-Small,EdgeMLP+CatBoost,time_70_15_15,0.968614,0.963061,...,0.001016,226.864111,226.616421,450000,450000
